# Notebook 6: Molecular docking

In notebook 5 you asked a *geometric* question of every compound in your database: can this
molecule place three hydrophobic groups the way p53 does? The pharmacophore that answered it knew
the positions of the three hot spots, but it knew nothing else about MDM2 - not the shape of the
walls between the subpockets, not the hydrogen-bonding groups that line them, and nothing at all
about whether the molecule would actually *fit*.

In this notebook you add the method that does use the whole protein: **molecular docking**. Docking
takes a ligand and a receptor structure and tries to answer two questions at once. *Where* does the
molecule sit, and *how good* is that arrangement? The first is a search problem over the position,
the orientation and the rotatable bonds of the ligand; the second needs a **scoring function** that
puts a number on a protein-ligand complex. You will see that the search is the easier half.

The notebook is also an honest one. We will dock a ligand back into the crystal structure it came
from - the standard way to find out whether a docking protocol works at all - and the protocol will
**fail**. Working out *why* it fails is the most useful thing in this notebook, because that
diagnosis is the skill that separates a docking campaign that means something from one that
produces a nicely ranked table of nonsense.

## Prepare yourself:

Read before the class. Both papers are open access, so the links go to the publishers directly:

- <a href="https://doi.org/10.1002/jcc.21334" target="_blank">Trott, O.; Olson, A.J. (2010) AutoDock Vina: improving the speed and accuracy of docking with a new scoring function, efficient optimization and multithreading. *J. Comput. Chem.* **31**, 455-461.</a> - the search algorithm and the scoring function you will be using.
- <a href="https://doi.org/10.1021/acs.jcim.1c00203" target="_blank">Eberhardt, J.; Santos-Martins, D.; Tillack, A.F.; Forli, S. (2021) AutoDock Vina 1.2.0: new docking methods, expanded force field, and Python bindings. *J. Chem. Inf. Model.* **61**, 3891-3898.</a> - the version used here, and the Python interface.

## What you will learn

At the end of this notebook you should be able to explain:

1. what docking is - a **search** over ligand poses combined with a **scoring function** - and how
   that differs from the Gaussian feature overlap of notebook 5;
2. what **receptor preparation** involves (chain and altloc selection, waters, ions, protonation,
   histidine tautomers) and why it is the step that most often ruins a docking campaign;
3. why docking needs a **defined binding site**, and how a search box is derived from a
   crystallographic ligand;
4. how to **validate** a docking protocol by **redocking**, how to measure the RMSD to the
   experimental pose, and how to distinguish a *search* failure from a *scoring* failure;
5. why a docking score is a **weak predictor of affinity**, and what docking is nevertheless good
   for;
6. how docking refines the candidate set that notebooks 4 and 5 produced, and what would be needed
   before anyone would claim that one of those compounds binds.

Roughly a third of the notebook is concepts and two thirds are hands-on.

## Planning (approximately 2 hours)

| Section | Content | Time | of which computing |
|---|---|---|---|
| 1 | What docking is, and how it differs from a pharmacophore search | 15 min | - |
| 2 | Running this notebook on CalcUA: cores, storage, working offline | 10 min | ~2 min |
| 3 | The receptor: choosing 4HG7 and preparing it | 20 min | <1 min |
| 4 | The binding site and the search box | 10 min | <1 min |
| 5 | AutoDock Vina: the search, the score, `exhaustiveness` and the seed | 10 min | <1 min |
| 6 | Validation 1: redocking nutlin-3a, and why it fails | 25 min | ~2 min |
| 7 | Validation 2: the five known inhibitors against their potency | 15 min | ~1 min |
| 8 | Screening the candidates of notebooks 4 and 5 | 20 min | ~9 min |
| 9 | Analysing the hits and comparing the two screens | 15 min | <1 min |
| 10-12 | Discussion, exercises and take-home messages | 10 min | - |

**Read this before you start.** The computing times above are **measured**, not estimated: the whole
notebook executes in a little over **11 minutes** on **8 CPU cores**, of which the screen in section
8 is 8.9 minutes (116 compounds at 4.6 s each). On 4 cores the same screen takes about 13 minutes,
so the notebook fits the session either way - see section 2 for how the core count is chosen.

Only two cells are slow enough to notice: the `exhaustiveness` comparison in section 6.2 (75 s, and
it is the point of the section that it is wasted effort) and the screen in section 8. Both print
progress, and both are marked **SLOW** in the code. Section 8 is the one to start and then read on
while it runs.

## 1. What docking is

A docking program is given two things: a **receptor** structure, which it treats as a rigid body,
and a **ligand**, which it treats as a molecule with a fixed covalent geometry but freely rotatable
single bonds. It then does two things.

**It searches.** A *pose* is a complete specification of where the ligand is: three translations,
three rotations, and one torsion angle per rotatable bond. For a molecule with eight rotatable
bonds that is a fourteen-dimensional space, far too large to enumerate, so docking programs sample
it. AutoDock Vina uses a **Monte-Carlo based search**: from a random starting pose it repeatedly
makes a random change, locally minimises the result, and keeps or rejects it according to a
Metropolis criterion. One such chain is a *run*, and Vina performs several independent runs and
collects the best poses from all of them. This is a stochastic algorithm: two runs with different
random numbers do not give identical answers.

**It scores.** Every pose is given a number, and the number is what decides which pose is reported
and how the compounds in a screen are ranked. Vina's scoring function is **empirical**: a weighted
sum of simple geometric terms over all ligand-receptor atom pairs - three steric terms that depend
only on the distance between the atom surfaces (a Gaussian attraction, a second wider Gaussian, and
a repulsion), a hydrophobic term between apolar atoms, and a hydrogen-bonding term between donors
and acceptors - plus a penalty proportional to the number of rotatable bonds. The weights were
fitted to a set of complexes with measured affinities, which is why the result is reported in
kcal/mol and why it is tempting, and wrong, to read it as a binding free energy.

Notice what is *not* in that list: no solvent, no entropy beyond the torsion penalty, no
polarisation, no protein flexibility. A docking score is a fast, crude estimate, and section 7 will
show you exactly how crude.

### 1.1 How this differs from the pharmacophore search of notebook 5

The two methods look superficially similar - both align a 3D molecule against a 3D model of the
site - but they use different information and they fail in different ways.

| | pharmacophore search (notebook 5) | docking (this notebook) |
|---|---|---|
| what the model contains | three feature positions derived from p53 | all 911 atoms of the receptor |
| what is optimised | Gaussian overlap volume between features | an energy-like score over atom pairs |
| ligand flexibility | a pre-generated ensemble of rigid conformers | torsions are varied *during* the search |
| does it know the protein's shape? | only through optional excluded volumes | yes, that is most of the score |
| does it need a defined site? | **no** - the model *is* the site | **yes** - a search box must be given |
| what you get out | a score, and an aligned conformer | a score, and a predicted binding pose |
| cost per molecule | milliseconds | seconds |

The last two rows matter most in practice. A pharmacophore search is cheap enough to run on a
million compounds and tells you *whether* a molecule can present the right groups. Docking is a
thousand times slower, which is why it comes later in the funnel, but it produces a **pose**: a
concrete, falsifiable structural hypothesis that a chemist can look at and criticise.

**Question 1.** A pharmacophore search needs no binding site to be specified, while docking needs a
box. Explain where the information about "where the site is" sits in each of the two methods. What
would happen if you gave a docking program a box that covered the whole protein?

## 2. Running this notebook on CalcUA

This notebook is meant to be run on **CalcUA**, the University of Antwerp cluster of the Flemish
Supercomputer Centre (VSC). That environment is different from a laptop in four ways that the code
below has to respect, and all four are worth understanding because they come back in every
computational project you will do on shared hardware.

**You are not root.** You cannot install software system-wide. Everything here installs into your
own account, which `pip install --user` does by default on the cluster; in a virtual environment or
a conda environment the same `pip install` command writes into that environment instead. Either
route works - what matters is that nothing needs administrator rights.

**One version constraint, and it will bite you if you ignore it.** AutoDock Vina is the only package
here with compiled code. On PyPI it ships Linux x86_64 wheels for **Python 3.8 to 3.12 only** - there
is no wheel for 3.13 or 3.14, so on a newer Python `pip install vina` falls back to building from
source and fails. conda-forge does carry builds for 3.13 and 3.14. So:

| Your Python | How to install Vina |
|---|---|
| 3.8 - 3.12 | `pip install vina` (a wheel, no compiler needed) |
| 3.13 or newer | `conda install -c conda-forge vina`, or make a Python 3.12 environment |

The other notebooks of this course run on Python 3.13, so **this is the one notebook for which you
may need a separate environment**. On CalcUA the cleanest route is a conda environment of your own:

```
conda create -n docking -c conda-forge python=3.12 vina
conda activate docking
pip install meeko rdkit py3Dmol pandas numpy matplotlib tqdm jupyter
```

The next cell checks your Python version before it installs anything, and tells you which of the two
routes you need.

**A compute node may have no route to the internet.** Jobs that download their own input are a
classic way to waste an allocation: the job starts, the download hangs, the wall-clock limit
expires. This notebook therefore reads **every** input from a file in the course repository, using
the same local-copy-first pattern as `load_pdb()` in notebook 5. There is no live call to RCSB,
ChEMBL or PubChem anywhere in the critical path.

**The node is shared.** A login node is shared with everyone who is logged in, and a compute node
is shared with whatever else Slurm has placed on it. Taking all the cores because they happen to be
visible is antisocial and, on a login node, likely to get your process killed. The number of cores
you may use is the number Slurm *gave* you, which is in `$SLURM_CPUS_PER_TASK`. We read that, fall
back to the affinity mask of the process when it is absent, and cap the result so that an
interactive session on a login node stays modest. The number is then passed to Vina explicitly -
left to itself, Vina uses every core it can see.

**Home directories have quotas.** Everything this notebook writes goes into a `scratch/` folder, as
in notebooks 0-5. If your home directory is too small, set the environment variable
`NOTEBOOK6_SCRATCH` to somewhere on `$VSC_SCRATCH` before starting Jupyter and the notebook will
write there instead.

Nothing in this notebook needs a GPU.

In [ ]:
# Check the Python version before installing: there is no Vina wheel for
# Python 3.13 or newer, and a source build will fail (see the table above).
import sys

if sys.version_info[:2] > (3, 12):
    print(f"This kernel runs Python {sys.version.split()[0]}.")
    print("There is no AutoDock Vina wheel for this version on PyPI, so 'pip install vina'")
    print("would try to compile it and fail. Use conda-forge instead:")
    print()
    print("    conda create -n docking -c conda-forge python=3.12 vina")
    print("    conda activate docking")
    print("    pip install meeko rdkit py3Dmol pandas numpy matplotlib tqdm jupyter")
    print()
    print("then restart Jupyter with that environment's kernel and run this notebook again.")
else:
    print(f"Python {sys.version.split()[0]}: a Vina wheel is available, installing with pip.")

In [ ]:
# NOTE the % instead of the ! that the earlier notebooks use. "%pip" installs into the
# environment of *this kernel*; "!pip" runs whichever pip happens to be first on your
# PATH, which for this notebook is often a different Python - and then the install
# either fails or lands somewhere the kernel cannot see it.
#
# On CalcUA this installs into your own account:
#  - inside a virtual environment or a conda environment, into that environment;
#  - outside one, add --user  ->  %pip install --user vina meeko
# Skip this cell if you installed Vina with conda, as described above.
%pip install vina meeko
%pip install rdkit py3Dmol pandas numpy matplotlib tqdm

In [ ]:
# RDKit chemistry
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolAlign

# RDKit drawing
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import rdDepictor
IPythonConsole.ipython_useSVG = True
rdDepictor.SetPreferCoordGen(True)

# Numerical work, tables and plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 3D visualisation in the notebook (as in notebooks 1 and 5), file handling, timing
import py3Dmol
import os
import shutil
import sys
import time
from tqdm.auto import tqdm

# Docking: AutoDock Vina through its Python API, and Meeko to prepare the molecules
from vina import Vina
import meeko
from meeko import MoleculePreparation, PDBQTWriterLegacy
from meeko import PDBQTMolecule, RDKitMolCreate

# RDKit complains about every molecule it cannot sanitise; in a screen of
# thousands of compounds that is noise, so we silence it and count the failures
# ourselves instead.
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

from importlib.metadata import version as package_version

print("Python  :", sys.version.split()[0])
print("RDKit   :", Chem.rdBase.rdkitVersion)
print("Meeko   :", meeko.__version__)
print("Vina    :", package_version("vina"))

### 2.1 What machine are we actually on?

Rather than assuming, let us look. The cell below reports the things that decide how this notebook
should behave: the Python version, whether an Lmod module system and Slurm are present, whether the
VSC storage variables are set, and how many cores we are entitled to.

In [ ]:
# --- Where are we? Detect, do not assume. -----------------------------------
print("platform            :", sys.platform, os.uname().machine)
print("python              :", sys.version.split()[0])

# An Lmod module system sets these; "module" itself is a shell function, so it is
# not visible to shutil.which()
lmod = os.environ.get("LMOD_CMD") or os.environ.get("MODULESHOME")
print("module system       :", lmod if lmod else "not detected")

# Slurm: the scheduler commands, and the variables that exist only inside a job
slurm = shutil.which("sinfo") or shutil.which("sbatch")
print("slurm commands      :", slurm if slurm else "not found")
slurm_vars = {k: v for k, v in os.environ.items() if k.startswith("SLURM_")}
print("inside a slurm job  :", "yes" if slurm_vars else "no (login node or laptop)")

# VSC storage
for name in ("VSC_HOME", "VSC_DATA", "VSC_SCRATCH"):
    print(f"{name:20s}:", os.environ.get(name, "not set"))

In [ ]:
# --- How many cores may we use? --------------------------------------------
MAX_CPU = 8          # do not grab a whole node for a teaching notebook

def available_cpus(maximum=MAX_CPU):
    """Return the number of CPU cores this notebook is entitled to use.

    Inside a Slurm job the allocation is authoritative. Outside one we take the
    affinity mask of this process, which on a cluster login node is usually
    narrower than the physical core count. The result is capped, because a
    login node is shared with every other user who is logged in.
    """
    allocated = os.environ.get("SLURM_CPUS_PER_TASK")
    if allocated:
        n = int(allocated)
        source = "$SLURM_CPUS_PER_TASK"
    elif hasattr(os, "sched_getaffinity"):          # Linux
        n = len(os.sched_getaffinity(0))
        source = "affinity mask of this process"
    else:                                           # macOS, Windows
        n = os.cpu_count() or 1
        source = "os.cpu_count()"
    return min(n, maximum), n, source


N_CPU, n_visible, cpu_source = available_cpus()
print(f"cores visible ({cpu_source}): {n_visible}")
print(f"cores this notebook will use   : {N_CPU}   (capped at {MAX_CPU})")

# --- Where do we write? -----------------------------------------------------
# Everything generated by this notebook goes here. Set NOTEBOOK6_SCRATCH to a
# directory on $VSC_SCRATCH if your home directory is short of quota.
SCRATCH = os.environ.get("NOTEBOOK6_SCRATCH", "scratch")
os.makedirs(SCRATCH, exist_ok=True)
print(f"output directory               : {SCRATCH}")

# --- Reproducibility --------------------------------------------------------
RANDOM_SEED = 42
print(f"random seed                    : {RANDOM_SEED}")

The seed deserves a sentence of its own, because it is the difference between a result you can
defend and a result you cannot.

Vina's search is stochastic, and by default it takes its seed from the clock: run the same docking
twice and you get two slightly different answers, occasionally two *very* different answers. Passing
an explicit `seed=` makes a run reproducible, so that when a number in this notebook surprises you,
you can re-run the cell and know that any change you then make is the cause of a difference you see.
Note the limit of that guarantee: it fixes *this* calculation on *this* machine with *this* number
of cores. Vina seeds each of its parallel runs from the master seed, so changing the core count
changes how the work is divided and therefore changes the result. A seed makes a calculation
repeatable; it does not make it right, and a result that depends on the seed is telling you that
your search was too short.

## 3. The receptor

### 3.1 Why 4HG7, and not 1YCR

Notebooks 1 and 5 used **1YCR**, the MDM2-p53 complex. For docking it is the wrong choice, and the
reason is instructive.

1YCR is a 2.6 A structure of MDM2 with a **peptide** in the cleft. What a docking program needs is a
pocket shaped the way it is shaped when a *small molecule* is in it, and the two are not the same:
the MDM2 cleft adjusts to its ligand, most visibly at **Tyr100**, which sits at the mouth of the
Leu26 subpocket and acts as a lid.

That claim is easy to check, and worth checking rather than repeating. Measuring the Tyr100 side-chain
torsion chi1 (N-CA-CB-CG) in the structures this course has met gives:

| Structure | ligand in the cleft | Tyr100 chi1 |
|---|---|---|
| 1YCR | p53 peptide | **-162 degrees** |
| 4HG7 | nutlin-3a | -93 degrees |
| 5TRF | SAR405838 | -101 degrees |
| 5OC8 | siremadlin | -77 degrees |
| 4ZYF | NVP-CGM097 | -91 degrees |
| 4OAS | piperidinone | -90 degrees |

The peptide complex is in a different rotamer from **every** small-molecule complex, which agree
with each other to within about 25 degrees. Docking small molecules into 1YCR would mean docking into
a lid position that no small molecule has ever been observed to bind to.

We therefore use <a href="https://www.rcsb.org/structure/4HG7" target="_blank">**4HG7**</a>, which
is already in the repository from notebook 5. It is a good docking receptor for four reasons:

1. **Resolution 1.6 A**, R<sub>work</sub> 0.173, R<sub>free</sub> 0.215 - the best-resolved MDM2
   structure we have met in this course, so the side-chain positions in the pocket are reliable
   (recall the discussion of resolution and R-factors in notebook 1);
2. it is the **same pocket** that the whole course has been about, with a small molecule in it;
3. its crystallographic ligand is **nutlin-3a**, which is one of the five validation compounds of
   notebook 5, so the two notebooks can be compared directly;
4. that ligand does double duty: it **defines the search box** (section 4) and it makes
   **redocking validation** possible (section 6). A receptor without a bound ligand gives you no way
   to find out whether your protocol works.

1YCR keeps its own role: it is the structure that defines where the Phe19, Trp23 and Leu26 hot spots
are, and we will use it in section 4 to show that the box we build around the nutlin-3a ligand
really does cover those three subpockets.

One caveat that you should know about this entry rather than discover later: the crystallised
construct carries two surface mutations, **E69A and K70A**, introduced to help the protein
crystallise. Both are on the surface, 20 A or more from the ligand, so they do not affect the
pocket - but "the structure contains something that is not the wild-type protein" is exactly the
kind of detail that a `REMARK` record will tell you and that your docking program never will.

### 3.2 What receptor preparation means

A PDB file is not a docking input. Turning one into a receptor means making a series of decisions,
and this is where docking campaigns are usually lost - not in the docking itself. For 4HG7:

- **Which chain?** Only one here (chain A, residues 18-108). Easy. With a dimer in the asymmetric
  unit you would have to decide which copy, and whether the site lies at the interface.
- **Alternate conformations.** Nine residues in 4HG7 are modelled in two conformations. A docking
  program cannot accept two positions for the same atom, so one has to be chosen. We take altloc
  `A`, which is the major conformation. Of these residues only **Leu57** lines the pocket, at
  occupancy 0.8.
- **The waters.** 4HG7 contains 97 waters, three of which bridge the ligand to the protein. Deleting
  all waters means pretending the pocket is dry; keeping some means deciding *which*, and then
  asserting that they stay put when a different ligand binds. Neither choice is obviously right.
  We prepare **both** receptors, dock against the dry one, and use the hydrated one in section 6 to
  measure what that decision costs.
- **The ions and buffer components.** One half-occupancy sulfate sits at the rim of the pocket,
  3.5 A from the ligand. It is a crystallisation additive, not part of MDM2, and it is removed.
- **Protonation.** A PDB file from X-ray crystallography has no hydrogens (4HG7 has none), yet the
  hydrogen-bonding term of the scoring function needs to know which atoms are donors and which are
  acceptors. The standard states at pH 7 are uncontroversial for Lys, Arg, Asp and Glu. Histidine is
  not: it has two possible neutral tautomers, and the choice turns one ring nitrogen into a donor
  and the other into an acceptor. Both of MDM2's pocket histidines are involved, so we assign them
  from the crystal structure's own hydrogen-bonding geometry rather than letting a program guess
  (section 3.3).

### 3.3 The two histidines, decided from the crystal structure

His73 and His96 both line the pocket. Their tautomers were assigned by looking at what their ring
nitrogens are hydrogen-bonded to in 4HG7:

| Residue | N-delta1 contact | N-epsilon2 contact | Assignment |
|---|---|---|---|
| His73 | sulfate O4 at **2.57 A** | water at 2.71 A | **HID** - N-delta1 protonated, donating to the sulfate |
| His96 | water at 2.85 A | Val93 backbone O at **3.26 A**, water at 2.70 A | **HIE** - N-epsilon2 protonated, donating to the Val93 carbonyl |

Neither assignment is forced by the data - a water can accept from either nitrogen - but each is the
more consistent reading, and writing the reasoning down is what makes the choice reviewable. The
His73 case comes with a small irony: the contact that decides its tautomer is a sulfate ion that we
then delete as a crystallisation artefact.

**Question 2.** Receptor preparation is often described as the step where docking campaigns are won
or lost. Pick two of the decisions above - the waters, the alternate conformations, the sulfate, the
histidine tautomers - and describe what would go wrong if you made each of them the other way.
Which of the two would you expect to change the docking scores more?

### 3.4 Loading the prepared receptor

Preparation is fiddly and not very educational to watch, so it was done once and the result is
committed to the repository. `assets/databases/notebook-6/README.md` records exactly what was done,
and the cell after next regenerates the files if you want to check. Here we just load them.

In [ ]:
PDB_DIR = os.path.join("assets", "databases", "notebook-5")      # 4HG7.pdb, 1YCR.pdb
NB6_DIR = os.path.join("assets", "databases", "notebook-6")      # the prepared receptors

RECEPTOR_DRY = os.path.join(NB6_DIR, "4HG7_receptor.pdbqt")
RECEPTOR_WET = os.path.join(NB6_DIR, "4HG7_receptor_waters.pdbqt")


def load_pdb(pdb_id):
    """Return the text of a PDB file from the course repository.

    The same pattern as in notebook 5: the local copy comes first, so that this
    notebook also runs on a compute node without a route to the internet. Only
    if no local copy exists at all is the structure fetched from the RCSB PDB.
    """
    course_copy = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    if os.path.exists(course_copy):
        print(f"{pdb_id}: using the local copy in {PDB_DIR}")
        with open(course_copy) as f:
            return f.read()

    scratch_copy = os.path.join(SCRATCH, f"{pdb_id}.pdb")
    if os.path.exists(scratch_copy):
        print(f"{pdb_id}: using the copy downloaded earlier, {scratch_copy}")
        with open(scratch_copy) as f:
            return f.read()

    print(f"{pdb_id}: not in the repository - downloading from the RCSB PDB")
    import requests
    text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb").text
    with open(scratch_copy, "w") as f:
        f.write(text)
    return text


pdb_4hg7 = load_pdb("4HG7")

for path in (RECEPTOR_DRY, RECEPTOR_WET):
    with open(path) as f:
        atoms = [line for line in f if line.startswith("ATOM")]
    waters = sum(1 for line in atoms if line[17:20] == "HOH")
    print(f"{os.path.basename(path):28s} {len(atoms):4d} atoms, of which {waters} water atoms")

A PDBQT file is a PDB file with two extra columns: a partial charge and an **AutoDock atom type**.
The types are what the scoring function works with - `C` for apolar carbon, `A` for aromatic carbon,
`N` and `OA` for a nitrogen and an acceptor oxygen, `HD` for a hydrogen attached to a donor, and so
on. Note that only the **polar** hydrogens are present: the apolar ones are merged into the carbon
they sit on, which is the united-atom convention AutoDock has always used. Let us look at a few
lines and count the types.

In [ ]:
with open(RECEPTOR_DRY) as f:
    receptor_lines = [line.rstrip("\n") for line in f if line.startswith("ATOM")]

print("The first five atoms of the prepared receptor:\n")
for line in receptor_lines[:5]:
    print(line)

types = pd.Series([line[77:79].strip() for line in receptor_lines]).value_counts()
print("\nAutoDock atom types in the receptor:")
for atom_type, n in types.items():
    print(f"  {atom_type:3s} {n:4d}")

### 3.5 Optional: regenerating the receptor files

This cell reproduces both PDBQT files from `4HG7.pdb` with the Meeko Python API, and checks that the
result is byte-for-byte identical to what is in the repository. You do not need to run it - it is
here so that the preparation is auditable rather than a black box. The equivalent command line is in
`assets/databases/notebook-6/README.md`.

In [ ]:
# OPTIONAL - the committed receptor files were made exactly like this.
REGENERATE_RECEPTOR = False       # set to True to run it

if REGENERATE_RECEPTOR:
    from meeko import Polymer, ResidueChemTemplates

    # The three waters that bridge nutlin-3a to the protein (see section 6.3)
    BRIDGING_WATERS = (307, 327, 338)

    protein = [l for l in pdb_4hg7.splitlines() if l.startswith("ATOM")]
    waters = [l for l in pdb_4hg7.splitlines()
              if l.startswith("HETATM") and l[17:20] == "HOH"
              and int(l[22:26]) in BRIDGING_WATERS]
    # Dropped on purpose: the ligand NUT 201, the sulfate SO4 202, and the other 94 waters

    for name, lines in (("4HG7_receptor.pdbqt", protein),
                        ("4HG7_receptor_waters.pdbqt", protein + waters)):
        polymer = Polymer.from_pdb_string(
            "\n".join(lines) + "\nEND\n",
            ResidueChemTemplates.create_from_defaults(),
            MoleculePreparation(),
            set_template={"A:73": "HID", "A:96": "HIE"},   # see section 3.3
            default_altloc="A",                            # the major conformation
        )
        rigid_pdbqt, _flex = PDBQTWriterLegacy.write_from_polymer(polymer)
        out = os.path.join(SCRATCH, name)
        with open(out, "w") as f:
            f.write(rigid_pdbqt)
        with open(os.path.join(NB6_DIR, name)) as f:
            same = f.read() == rigid_pdbqt
        print(f"{name}: written to {out}, identical to the committed file: {same}")
else:
    print("Skipped. Set REGENERATE_RECEPTOR = True to rebuild the receptor files.")

## 4. The binding site and the search box

Docking needs to be told where to look. The region it searches is a rectangular box, given by a
centre and three side lengths, and everything outside it is invisible to the calculation: Vina
precomputes its scoring terms on a grid over the box, and a pose is only scored where that grid
exists.

The box size is a real trade-off. Too small and you forbid the correct answer - the ligand cannot
reach the pose it should have, or its substituents are pushed out of the grid. Too large and you
waste the search: the volume to be explored grows as the cube of the side length, so with a fixed
`exhaustiveness` each cubic angstrom gets less attention and the chance of missing the right pose
goes *up*. A box that encloses the site with a few angstroms to spare in each direction is the usual
compromise.

The honest way to place the box here is to derive it from the crystallographic ligand: take the
bounding box of the nutlin-3a atoms and add a margin. This is standard practice, and it is worth
being explicit about what it assumes - that we already know where the site is. That assumption is
fine in this course, because we have a co-crystal structure. For a target with no bound ligand you
would have to find the site first, with a pocket-detection program or from the conservation of the
sequence, and that is a separate problem with its own failure modes.

In [ ]:
LIGAND_RESNAME = "NUT"       # nutlin-3a, the crystallographic ligand of 4HG7
BOX_MARGIN = 5.0             # angstroms added on each side of the ligand's bounding box

# The ligand atoms, straight out of the PDB file
ligand_pdb_lines = [l for l in pdb_4hg7.splitlines()
                    if l.startswith("HETATM") and l[17:20] == LIGAND_RESNAME]
ligand_xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                       for l in ligand_pdb_lines])

BOX_CENTER = ligand_xyz.mean(axis=0)
BOX_SIZE = (ligand_xyz.max(axis=0) - ligand_xyz.min(axis=0)) + 2 * BOX_MARGIN

print(f"{len(ligand_xyz)} atoms in the crystallographic ligand {LIGAND_RESNAME}")
print("ligand extent        : %.1f x %.1f x %.1f A" % tuple(ligand_xyz.max(axis=0) - ligand_xyz.min(axis=0)))
print("box centre           : %.2f, %.2f, %.2f" % tuple(BOX_CENTER))
print("box size (+%.0f A each side): %.1f x %.1f x %.1f A" % ((BOX_MARGIN,) + tuple(BOX_SIZE)))
print("box volume           : %.0f A^3" % np.prod(BOX_SIZE))

### 4.1 Does the box cover the three hot spots?

A sanity check that costs nothing and catches a whole class of mistakes. The box was built around a
small molecule; the pocket we care about was defined in notebooks 1 and 5 by three p53 side chains
in a *different* structure. Do they coincide?

To find out we superpose 1YCR onto 4HG7 on their MDM2 backbone atoms, and then ask where the Phe19,
Trp23 and Leu26 side chains of p53 end up relative to our box.

In [ ]:
pdb_1ycr = load_pdb("1YCR")


def ca_coordinates(pdb_text, chain):
    """Return {residue number: coordinates} for the C-alpha atoms of one chain."""
    out = {}
    for line in pdb_text.splitlines():
        if not line.startswith("ATOM") or line[21] != chain:
            continue
        if line[12:16] != " CA " or line[16] not in " A":    # skip minor altlocs
            continue
        out[int(line[22:26])] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return out


def superpose(mobile, target):
    """Kabsch superposition. Returns a function that maps mobile coordinates onto target."""
    common = sorted(set(mobile) & set(target))
    P = np.array([mobile[i] for i in common])
    Q = np.array([target[i] for i in common])
    Pc, Qc = P.mean(axis=0), Q.mean(axis=0)
    U, S, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    rmsd = np.sqrt(np.mean(np.sum(((R @ (P - Pc).T).T + Qc - Q) ** 2, axis=1)))
    return (lambda xyz: (R @ (np.atleast_2d(xyz) - Pc).T).T + Qc), rmsd, len(common)


# MDM2 is chain A in both entries; p53 is chain B of 1YCR
transform, fit_rmsd, n_fitted = superpose(ca_coordinates(pdb_1ycr, "A"),
                                         ca_coordinates(pdb_4hg7, "A"))
print(f"1YCR superposed on 4HG7: C-alpha RMSD {fit_rmsd:.2f} A over {n_fitted} residues\n")

# Where do the three p53 hot-spot side chains land?
HOTSPOT_ATOMS = {19: ["CG", "CD1", "CD2", "CE1", "CE2", "CZ"],                        # Phe19
                 23: ["CG", "CD1", "NE1", "CE2", "CD2", "CE3", "CZ2", "CZ3", "CH2"],  # Trp23
                 26: ["CB", "CG", "CD1", "CD2"]}                                      # Leu26
LABELS = {19: "Phe19", 23: "Trp23", 26: "Leu26"}

box_lo, box_hi = BOX_CENTER - BOX_SIZE / 2, BOX_CENTER + BOX_SIZE / 2
hotspot_in_frame = {}
for resnum, names in HOTSPOT_ATOMS.items():
    xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                    for l in pdb_1ycr.splitlines()
                    if l.startswith("ATOM") and l[21] == "B"
                    and int(l[22:26]) == resnum and l[12:16].strip() in names])
    moved = transform(xyz)
    hotspot_in_frame[resnum] = moved
    inside = np.all((moved >= box_lo) & (moved <= box_hi), axis=1)
    d = np.linalg.norm(moved.mean(axis=0) - BOX_CENTER)
    print(f"  {LABELS[resnum]}: {inside.sum()}/{len(moved)} side-chain atoms inside the box, "
          f"centroid {d:.1f} A from the box centre")

### 4.2 Looking at it

Now put all of it in one picture: the MDM2 surface, the crystallographic nutlin-3a, the search box,
the three p53 hot-spot positions and the pocket residues. Rotate it and convince yourself that the
box is a sensible description of the site - that it encloses the cleft, that it does not extend far
into the solvent, and that the ligand and the three hot spots occupy the same volume.

In [ ]:
view = py3Dmol.view(width=800, height=550)

# model 0: the whole 4HG7 entry
view.addModel(pdb_4hg7, "pdb")
view.setStyle({"model": 0}, {})                                   # start from nothing
view.setStyle({"model": 0, "hetflag": False}, {"cartoon": {"color": "lightgray"}})
view.addSurface(py3Dmol.VDW, {"opacity": 0.60, "color": "lightgray"},
                {"model": 0, "hetflag": False})

# the pocket residues, as sticks
POCKET_RESIDUES = [54, 57, 58, 61, 62, 67, 72, 73, 75, 91, 93, 96, 99, 100]
view.addStyle({"model": 0, "resi": POCKET_RESIDUES},
              {"stick": {"colorscheme": "whiteCarbon", "radius": 0.15}})

# the crystallographic ligand, as yellow sticks
view.addStyle({"model": 0, "resn": LIGAND_RESNAME},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}})

# model 1: the three p53 hot-spot side chains, superposed from 1YCR, as red spheres
hotspot_block = ""
serial = 1
for resnum, coords in hotspot_in_frame.items():
    for x, y, z in coords:
        hotspot_block += (f"HETATM{serial:5d}  C   HOT B{resnum:4d}    "
                          f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C\n")
        serial += 1
view.addModel(hotspot_block + "END\n", "pdb")
view.setStyle({"model": 1}, {"sphere": {"radius": 0.55, "color": "red", "opacity": 0.85}})

# the search box
view.addBox({"center": {"x": float(BOX_CENTER[0]), "y": float(BOX_CENTER[1]), "z": float(BOX_CENTER[2])},
             "dimensions": {"w": float(BOX_SIZE[0]), "h": float(BOX_SIZE[1]), "d": float(BOX_SIZE[2])},
             "color": "cyan", "opacity": 0.22})

view.zoomTo({"model": 0, "resn": LIGAND_RESNAME})
view.setBackgroundColor("white")
view.show()

Yellow is nutlin-3a as it sits in the crystal, the red spheres are the three p53 side chains that
notebooks 1 and 5 identified as the hot spots, brought here from 1YCR, and the cyan box is what
Vina will search. The two ligands - a peptide's three side chains and a small molecule - really do
fill the same cleft, which is the structural statement that the whole course rests on.

**Question 3.** The box printed above is about 20 x 23 x 19 A, some 8,300 A<sup>3</sup>, while the
ligand's own bounding box is only about 1,050 A<sup>3</sup> - the box is eight times the volume of
the thing we are placing in it. Suppose you halved the margin to 2.5 A, and suppose you doubled it
to 10 A. For each case, say what you would expect to happen to (a) the chance that the correct pose
is found at all and (b) the time the docking takes. Which of the two errors is harder to notice in a
table of docking scores? (Exercise 1 lets you check your answer.)

## 5. AutoDock Vina

**AutoDock Vina** is the docking program we will use - the most widely used open-source docking code,
originally published by Trott and Olson in 2010, and since version 1.2 available as a Python module
(Eberhardt *et al.*, 2021) rather than only as a command-line tool. We use the Python API, because
it lets us keep the receptor and its precomputed grids in memory and feed ligands to it one after
another, which is exactly what a virtual screen needs.

Three practical things to know before we start.

**Ligands have to be prepared too.** Vina does not read SMILES or SDF; it reads PDBQT, with explicit
atom types, partial charges and an explicit **torsion tree** that says which bonds may rotate and
which part of the molecule moves when they do. **Meeko** builds that from an RDKit molecule. One
consequence is easy to miss and important in section 6: Meeko freezes the ligand's bond lengths,
bond angles, ring conformations and non-rotatable torsions at whatever the input 3D structure had.
Vina never changes them. **The conformer you hand in is part of your docking protocol.**

**`exhaustiveness` buys confidence, not accuracy.** It is the number of independent Monte-Carlo runs
Vina performs, default 8. Each run is a fresh start, so more runs means a smaller chance that *all*
of them missed the best pose. It does not improve the scoring function: a thorough search of a
flawed landscape finds the flawed optimum more reliably. The practical consequence is that
`exhaustiveness` is the right knob for a *search* problem and useless for a *scoring* problem - and
section 6 is about telling the two apart. Vina parallelises over these runs, which is why the number
of cores and `exhaustiveness` interact.

**Scores are in kcal/mol, and are not free energies.** The number Vina reports is an empirical
score whose weights were fitted against measured affinities. It has the units and roughly the
magnitude of a binding free energy, and it correlates with affinity only weakly. Treat it as a
ranking device with a wide error bar.

The cell below sets up a Vina object with our receptor and our box. The grids are computed once and
reused for every ligand we dock afterwards.

In [ ]:
EXHAUSTIVENESS = 8      # number of independent Monte-Carlo runs; Vina's default
N_POSES = 9             # how many poses to keep per ligand


def make_docker(receptor_pdbqt, cpu=None, seed=RANDOM_SEED, exhaustiveness=EXHAUSTIVENESS):
    """Return a Vina object with the receptor loaded and the grids precomputed.

    cpu is passed explicitly: left to itself Vina uses every core it can see,
    which on a shared machine is not ours to take (see section 2).
    """
    v = Vina(sf_name="vina", cpu=cpu or N_CPU, seed=seed, verbosity=0)
    v.set_receptor(receptor_pdbqt)
    v.compute_vina_maps(center=BOX_CENTER.tolist(), box_size=BOX_SIZE.tolist())
    return v


t0 = time.time()
docker = make_docker(RECEPTOR_DRY)
print(f"receptor loaded and grids computed in {time.time() - t0:.1f} s")
print(docker)

### 5.1 Preparing a ligand

Two steps, and both can fail, so both are wrapped in a function that returns `None` rather than
raising. In a screen of thousands of compounds a handful will always be unparseable, fail to embed,
or carry an element the atom typer does not know, and the screen has to survive them.

1. **3D structure.** RDKit's ETKDGv3 distance-geometry embedding followed by MMFF minimisation -
   the same procedure as in notebooks 4 and 5, with the seed fixed.
2. **PDBQT.** Meeko perceives the rotatable bonds, assigns AutoDock atom types and Gasteiger
   charges, and writes the torsion tree.

In [ ]:
def prepare_ligand(smiles, seed=RANDOM_SEED, n_conformers=1, conformer_id=0):
    """SMILES -> (PDBQT string, RDKit molecule with hydrogens and 3D coordinates).

    Returns (None, None) with a reason when anything fails, so that a screen can
    carry on. n_conformers > 1 embeds an ensemble and uses conformer number
    conformer_id for the PDBQT.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, "SMILES could not be parsed"

    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if n_conformers == 1:
        if AllChem.EmbedMolecule(mol, params) != 0:
            return None, None, "3D embedding failed"
        AllChem.MMFFOptimizeMolecule(mol)
    else:
        if len(AllChem.EmbedMultipleConfs(mol, n_conformers, params)) == 0:
            return None, None, "3D embedding failed"
        AllChem.MMFFOptimizeMoleculeConfs(mol)

    try:
        setups = MoleculePreparation().prepare(mol, conformer_id=conformer_id)
        pdbqt, ok, error = PDBQTWriterLegacy.write_string(setups[0])
    except Exception as exc:                      # Meeko raises on exotic chemistry
        return None, None, f"Meeko failed: {exc}"
    if not ok:
        return None, None, f"Meeko refused the molecule: {error}"
    return pdbqt, mol, None


def poses_to_mol(vina_object, n_poses=N_POSES):
    """Read the poses out of a Vina object as one RDKit molecule with n conformers."""
    pdbqt_poses = vina_object.poses(n_poses=n_poses)
    pdbqt_mol = PDBQTMolecule(pdbqt_poses, skip_typing=True)
    return RDKitMolCreate.from_pdbqt_mol(pdbqt_mol)[0]


# Nutlin-3a, to see what comes out. The SMILES are taken verbatim from notebook 5.
NUTLIN_SMILES = "COc1ccc(c(OC(C)C)c1)C2=N[C@H]([C@H](N2C(=O)N3CCNC(=O)C3)c4ccc(Cl)cc4)c5ccc(Cl)cc5"

nutlin_pdbqt, nutlin_3d, err = prepare_ligand(NUTLIN_SMILES)
print("preparation error:", err)

torsions = [l for l in nutlin_pdbqt.splitlines() if l.startswith("TORSDOF")]
print("\nthe end of the PDBQT file Meeko produced:")
for line in nutlin_pdbqt.splitlines()[-4:]:
    print(" ", line)
print(f"\nrotatable bonds Vina will search over: {torsions[0].split()[1]}")
print(f"RDKit's own count of rotatable bonds : {Descriptors.NumRotatableBonds(Chem.MolFromSmiles(NUTLIN_SMILES))}")

Eight torsions plus six degrees of freedom for the rigid-body placement: the search space for this
one molecule is fourteen-dimensional. That is what `exhaustiveness` has to cover, and it is why
docking costs seconds per molecule rather than milliseconds.

**Question 4.** `exhaustiveness` defaults to 8. Suppose you raise it to 64 and the top-scoring pose
does not change. Suppose instead that the top-scoring pose *does* change. What do you learn in each
case, and which of the two outcomes would make you trust your docking more? How would you use this
to choose a value for a screen of 100,000 compounds?

## 6. Validation 1: redocking

Before a docking protocol is used to judge compounds nobody has measured, it has to be shown to
reproduce something that *is* known. The standard test is **redocking**: take the ligand out of a
co-crystal structure, throw its coordinates away, dock it back into its own receptor, and compare
the predicted pose with the experimental one.

The comparison is made with the **RMSD** between the heavy atoms of the predicted and the
experimental pose, in angstroms. The conventional threshold is **2 A**: below it, the pose is
usually considered to reproduce the crystal structure, and the protocol is considered acceptable for
that site. The threshold is a convention, not a physical constant, and it is lenient - at 2 A the
rings can be in rather different places.

Two warnings about this test, both of which matter for how you read the result.

It is the **easiest possible test**, and it is a necessary rather than a sufficient condition. The
receptor is the one that was solved *with this ligand in it*, so every side chain is already in the
right rotamer, and the pocket is shaped exactly to fit. If a protocol cannot reproduce a pose under
those circumstances, nothing it says about other compounds is worth much. But passing the test does
not prove that it will get a different chemical series right.

And the RMSD must be computed **without superposing** the two molecules, because the question is
whether the ligand is in the right place in the pocket, not whether it has the right shape. RDKit's
`GetBestRMS()` superposes; `CalcRMS()` does not. We want `CalcRMS()`. It does still handle
**molecular symmetry**, which we need: nutlin-3a has two 4-chlorophenyl rings and a *para*-
substituted ring can be flipped, so the same physical pose corresponds to several atom numberings,
and a naive atom-by-atom RMSD would report a large value for a perfect pose.

In [ ]:
# --- the experimental pose -------------------------------------------------
# A PDB file has no bond orders, so they are taken from the SMILES of notebook 5.
crystal_pose = Chem.MolFromPDBBlock("\n".join(ligand_pdb_lines) + "\nEND\n",
                                    sanitize=False, removeHs=False)
crystal_pose = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(NUTLIN_SMILES),
                                                    crystal_pose)
Chem.SanitizeMol(crystal_pose)

# Is the molecule in the crystal really the molecule whose SMILES we are using?
# The InChIKey settles it, stereochemistry included.
Chem.AssignStereochemistryFrom3D(crystal_pose)
key_crystal = Chem.MolToInchiKey(crystal_pose)
key_smiles = Chem.MolToInchiKey(Chem.MolFromSmiles(NUTLIN_SMILES))
print("InChIKey of the 4HG7 ligand   :", key_crystal)
print("InChIKey of the notebook-5 SMILES:", key_smiles)
print("same molecule, same stereochemistry:", key_crystal == key_smiles)


def pose_rmsd(docked, reference, conf_id=0):
    """Symmetry-corrected heavy-atom RMSD, without superposition.

    CalcRMS maps the atoms of the two molecules onto each other allowing for
    symmetry, but leaves the coordinates where they are - which is what we want:
    the question is whether the ligand is in the right place in the pocket.
    """
    return rdMolAlign.CalcRMS(Chem.RemoveHs(docked), Chem.RemoveHs(reference), prbId=conf_id)

### 6.1 The redocking run

Everything is in place: the receptor with its grids, the box, and a freshly generated 3D structure
of nutlin-3a that knows nothing about the crystal. Dock it, and compare all nine returned poses with
the experimental one.

In [ ]:
t0 = time.time()
docker.set_ligand_from_string(nutlin_pdbqt)
docker.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
redock_time = time.time() - t0

energies = docker.energies(n_poses=N_POSES)
redocked = poses_to_mol(docker)

rows = []
for i in range(redocked.GetNumConformers()):
    rows.append({"pose": i + 1,
                 "score (kcal/mol)": energies[i][0],
                 "RMSD to crystal (A)": round(pose_rmsd(redocked, crystal_pose, conf_id=i), 2)})
redock_df = pd.DataFrame(rows)

print(f"redocking took {redock_time:.1f} s on {N_CPU} cores "
      f"(exhaustiveness {EXHAUSTIVENESS}, seed {RANDOM_SEED})\n")
redock_df

In [ ]:
best = redock_df.iloc[0]
closest = redock_df.loc[redock_df["RMSD to crystal (A)"].idxmin()]

print(f"The best-scoring pose (pose {int(best['pose'])}) scores "
      f"{best['score (kcal/mol)']:.2f} kcal/mol and is "
      f"{best['RMSD to crystal (A)']:.2f} A from the crystal pose.")
print(f"The pose closest to the crystal (pose {int(closest['pose'])}) is "
      f"{closest['RMSD to crystal (A)']:.2f} A away and scores "
      f"{closest['score (kcal/mol)']:.2f} kcal/mol.\n")

if best["RMSD to crystal (A)"] <= 2.0:
    print("=> below 2 A: the protocol reproduces the experimental pose.")
else:
    print(f"=> {best['RMSD to crystal (A)']:.1f} A is far above the 2 A threshold.")
    print("   This redocking has FAILED. Sections 6.2 to 6.5 work out why.")

### The redocking fails

This is not the result a tutorial is supposed to produce, so let us be precise about what it says.
The best-scoring pose is nowhere near the crystal pose - it is at the wrong end of the cleft - and
*none* of the nine poses reaches 2 A. By the standard criterion this protocol is not fit to be used,
and if we had skipped this validation and gone straight to screening, nothing would have warned us.

There are only so many things that can be wrong, and they split into two groups:

- a **search** failure: the correct pose is a minimum of the scoring function, but the Monte-Carlo
  search did not find it. Causes: too few runs (`exhaustiveness`), a box that is too large, bad
  luck with the seed.
- a **scoring** failure: the search did its job, but the scoring function does not consider the
  correct pose to be the best one. Causes: a missing interaction (a bridging water, a metal), the
  wrong protonation, a rigid receptor, or simply the limits of an empirical function with five
  terms.

These need completely different repairs, and the difference is diagnosable. We do it in three steps.

### 6.2 Is it the search? Spend more effort on it

If the search is the problem, raising `exhaustiveness` should fix it. This cell is the first slow
one: three docking runs, the last of which does 64 Monte-Carlo runs. **Start it and read on.**

In [ ]:
# SLOW: about 75 s on 8 cores, two to three minutes on 4.
exhaustiveness_rows = []
for exh in (8, 32, 64):
    t0 = time.time()
    docker.set_ligand_from_string(nutlin_pdbqt)
    docker.dock(exhaustiveness=exh, n_poses=N_POSES)
    elapsed = time.time() - t0
    e = docker.energies(n_poses=N_POSES)
    poses = poses_to_mol(docker)
    rmsds = [pose_rmsd(poses, crystal_pose, conf_id=i) for i in range(poses.GetNumConformers())]
    exhaustiveness_rows.append({
        "exhaustiveness": exh,
        "best score": e[0][0],
        "RMSD of best-scoring pose (A)": round(rmsds[0], 2),
        "best RMSD among the 9 poses (A)": round(min(rmsds), 2),
        "time (s)": round(elapsed, 1),
    })
    print(f"  exhaustiveness {exh:3d}: {elapsed:5.1f} s, "
          f"best score {e[0][0]:6.2f}, RMSD {rmsds[0]:.2f} A", flush=True)

pd.DataFrame(exhaustiveness_rows)

Eight times the effort changes almost nothing. The score of the best pose barely moves and it is the
same wrong pose. That is the signature of a **converged search**: Vina is reliably finding the
minimum of its own scoring function, and the minimum of its scoring function is not the crystal
structure.

This is an important negative result, and it is the reason the diagnosis has to come before the
repair. Had we simply cranked `exhaustiveness` up for the whole screen, we would have paid eight
times the computer time for the same wrong answers.

### 6.3 Is it the score? Ask the scoring function about the experimental pose

The direct test. Put the ligand into the pocket *in its crystallographic pose* and score it, without
searching at all. Vina's `score()` does exactly that, and `optimize()` then does a local
minimisation, which tells us whether the crystal pose is even a local minimum of the score.

If the crystal pose scored much *better* than the best docked pose, the search would be at fault
after all. If it scores the same or worse, the scoring function is.

In [ ]:
# Score the experimental pose itself. It needs hydrogens (added at the
# crystallographic heavy-atom positions) and a PDBQT, but no embedding.
crystal_h = Chem.AddHs(crystal_pose, addCoords=True)
crystal_setup = MoleculePreparation().prepare(crystal_h)[0]
crystal_pdbqt, ok, err = PDBQTWriterLegacy.write_string(crystal_setup)
assert ok, err

docker.set_ligand_from_string(crystal_pdbqt)
score_crystal = docker.score()[0]
score_crystal_min = docker.optimize()[0]

print(f"score of the experimental pose           : {score_crystal:7.2f} kcal/mol")
print(f"  after local minimisation               : {score_crystal_min:7.2f} kcal/mol")
print(f"score of the best pose Vina found        : {best['score (kcal/mol)']:7.2f} kcal/mol")
print()
if best["score (kcal/mol)"] <= score_crystal:
    print("The scoring function prefers the wrong pose, or rates it equally.")
    print("=> this is a SCORING failure, not a search failure.")
else:
    print("The experimental pose scores better than anything Vina found.")
    print("=> this is a SEARCH failure.")

There it is. The experimental pose and the wrong pose score within a few hundredths of a kcal/mol of
each other - far inside the error of an empirical scoring function - and the ranking goes the wrong
way. Vina's search is not the problem; its scoring function simply cannot tell these two
arrangements apart.

That is worth sitting with for a moment, because it is the central limitation of docking and it is
usually stated far too gently. For a well-behaved, rigid, 1.6 A complex of a well-studied target,
the scoring function is unable to distinguish the right answer from an answer that is 8 A away.

### 6.4 What is missing? The three waters

A scoring failure means something real is absent from the model. In this pocket there is an obvious
candidate: we deleted all 97 crystallographic waters, and three of them are not bystanders - they
sit **between** the ligand and the protein:

| Water | contacts in 4HG7 |
|---|---|
| HOH 307 | nutlin-3a imidazoline N (2.87 A), His96 N-epsilon2 (2.70 A), Val93 O (3.11 A) |
| HOH 327 | nutlin-3a piperazinone N3 (2.96 A), Gln59 N-epsilon2 (2.37 A) |
| HOH 338 | nutlin-3a carbonyl O3 (2.73 A) |

Each of these is a hydrogen bond the dry receptor cannot form. In a dry pocket the polar groups of
the ligand have nothing to bind to, and a pose that buries them uselessly costs nothing - so a wrong
pose becomes competitive.

This is where the second receptor file comes in. `4HG7_receptor_waters.pdbqt` is the same protein
with exactly those three waters kept as part of the receptor. Let us ask the same question of it.

In [ ]:
docker_wet = make_docker(RECEPTOR_WET)
docker_wet.set_ligand_from_string(crystal_pdbqt)
score_crystal_wet = docker_wet.score()[0]

print("score of the EXPERIMENTAL pose")
print(f"  dry receptor                 : {score_crystal:7.2f} kcal/mol")
print(f"  with the three bridging waters: {score_crystal_wet:7.2f} kcal/mol")
print(f"  difference                   : {score_crystal_wet - score_crystal:7.2f} kcal/mol")

Roughly one kcal/mol, from three water molecules - and it is all hydrogen bonding that the dry
receptor threw away. One kcal/mol may not sound like much, but recall from section 6.3 that the
score difference between the right answer and an 8 A error was a few *hundredths* of a kcal/mol.
On that scale this is a large correction, and it points at a real omission: our receptor cannot make
three of the hydrogen bonds that the actual complex makes.

Including waters is not a free repair, though, and this is the uncomfortable part. Those waters were
located because *nutlin-3a* was in the pocket. Using them to dock other compounds asserts that they
stay in place whatever binds - and that is testable, so let us test it, using the other four
co-crystal structures of the validation compounds from notebook 5. (This comparison was done offline
and is reported here rather than computed, because it needs four more PDB entries.)

| Structure | ligand | HOH 307 | HOH 327 | HOH 338 |
|---|---|---|---|---|
| 5TRF | SAR405838 | **displaced** (ligand atom 1.8 A away) | water present 2.9 A away | **displaced** (2.7 A) |
| 5OC8 | siremadlin | water present 2.1 A away | water present 1.9 A away | water present 3.8 A away |
| 4ZYF | NVP-CGM097 | water present 1.1 A away | water present 2.7 A away | **no water nearby** |
| 4OAS | piperidinone series | **displaced** (0.6 A) | water present 2.0 A away | water present 2.8 A away |

The pattern is exactly the one that makes structure-based design hard: of three waters that are
essential for *this* ligand, each is displaced by at least one other inhibitor. A receptor with the
waters in it would score those compounds as clashing. **There is no single right answer**, which is
why we keep the waters as a diagnostic and dock against the dry receptor - the conservative choice,
which never forbids a pose that is actually possible.

### 6.5 The other half: the conformer you hand in

The waters are not the whole story. Recall from section 5 that Meeko freezes everything except the
rotatable torsions, so the ring conformations and the non-rotatable geometry of our ETKDG structure
are imposed on the docking. Nutlin-3a has a five-membered dihydroimidazole ring carrying both
chlorophenyl groups, and the conformation of that ring decides the relative orientation of the two
rings that must go into two of the three subpockets. Vina cannot fix it.

So let us dock the ligand once more, taking its internal geometry from the crystal structure while
still letting Vina search over all six rigid-body degrees of freedom and all eight torsions. Vina
randomises the pose before it starts, so this is a genuine search - it just starts from the right
ring conformation.

In [ ]:
# Internal geometry from the crystal, but the pose itself is searched from scratch.
geometry_rows = []
for label, receptor in (("dry receptor", docker), ("with the three waters", docker_wet)):
    receptor.set_ligand_from_string(crystal_pdbqt)
    receptor.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
    e = receptor.energies(n_poses=N_POSES)
    poses = poses_to_mol(receptor)
    rmsds = [pose_rmsd(poses, crystal_pose, conf_id=i) for i in range(poses.GetNumConformers())]
    geometry_rows.append({
        "receptor": label,
        "best score": e[0][0],
        "RMSD of best-scoring pose (A)": round(rmsds[0], 2),
        "best RMSD among the 9 poses (A)": round(min(rmsds), 2),
        "rank of that pose": int(np.argmin(rmsds)) + 1,
    })

print("Docking with the ligand's internal geometry taken from the crystal structure:\n")
pd.DataFrame(geometry_rows)

With the crystallographic ring conformation **and** the three waters, the best-scoring pose
reproduces the crystal structure to about half an angstrom - a textbook redocking result. With the
crystallographic ring conformation and a dry receptor, the native pose is found by the search and
sits a few places down the ranking: the search can reach it, but the dry scoring function will not
put it first.

So the failure had two causes, and we can now state both:

1. **The receptor was missing three bridging waters**, worth about a kcal/mol of hydrogen bonding in
   the native pose - large compared with the hundredths of a kcal/mol that separated right from
   wrong - without which a wrong pose scores as well as the right one.
2. **The ligand's internal geometry came from a conformer generator, not from the crystal.** Vina
   never changes it, so a ring conformation that ETKDG happened to produce was imposed on the whole
   docking.

Two things that are *not* to blame, and we showed it rather than assumed it: the amount of search
effort (section 6.2) and the size of the box (you can check this yourself in exercise 1).

The second cause is the more sobering one, because in a real screen you have **no crystal
conformation to fall back on** - that is the whole point of a prospective screen. The honest
conclusion is that our protocol has a known weakness that we cannot repair for unknown compounds,
and that is a limitation to carry into the rest of the notebook rather than a bug to fix. The
standard mitigation is to dock several starting conformers per molecule and keep the best, which is
exercise 2.

A third candidate cause deserves a mention, because it is the one the MDM2 literature talks about
most and we have *not* addressed it at all: the receptor is **rigid**. In this test that costs us
little, since the receptor was solved with this very ligand in it, so its side chains are already in
the right positions. For any other compound it is a real limitation.

**Question 5.** The experimental pose scored about the same as a pose 8 A away, and raising
`exhaustiveness` eightfold changed nothing. Explain in your own words why those two observations
together rule out a search failure. If you had only the second observation, what else could explain
it?

**Question 6.** The receptor is held rigid throughout this notebook. Section 3.1 measured one side
chain that takes up a different rotamer in every small-molecule complex than in the peptide complex,
and that varies by 25 degrees even among the small-molecule complexes. Name it, and explain why a
rigid receptor is a particularly awkward assumption for *that* residue. Vina can treat chosen side
chains as flexible - what would you have to decide before switching that on, and what would it cost
you, both in computer time and in the reliability of the scores?

**Question 7.** We obtained a good redocking result only after giving the ligand its crystal
conformation. Explain why this means the redocking validation, as we finally passed it, is weaker
evidence than it looks. What would be the fair version of this test?

In [ ]:
# Look at the failure. Yellow is the crystal pose, cyan is the best-scoring pose
# from the redocking of section 6.1.
view = py3Dmol.view(width=800, height=550)

view.addModel(pdb_4hg7, "pdb")
view.setStyle({"model": 0}, {})
view.setStyle({"model": 0, "hetflag": False}, {"cartoon": {"color": "lightgray"}})
view.addSurface(py3Dmol.VDW, {"opacity": 0.55, "color": "lightgray"},
                {"model": 0, "hetflag": False})
view.addStyle({"model": 0, "resn": LIGAND_RESNAME},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.22}})

# the three bridging waters, as red spheres
view.addStyle({"model": 0, "resn": "HOH", "resi": [307, 327, 338]},
              {"sphere": {"radius": 0.45, "color": "red"}})

# the best-scoring docked pose
view.addModel(Chem.MolToPDBBlock(Chem.RemoveHs(redocked), confId=0), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})

view.zoomTo({"model": 0, "resn": LIGAND_RESNAME})
view.setBackgroundColor("white")
view.show()

Yellow is where nutlin-3a really sits, cyan is where Vina put it, and the red spheres are the three
waters we deleted. Note that the docked pose is not absurd - it is in the cleft, it is making
hydrophobic contacts, and nothing about it looks obviously wrong. **That** is why a pose always has
to be looked at next to something experimental, and why a plausible-looking pose from a screen of
unknown compounds carries so little weight.

## 7. Validation 2: the five known inhibitors

The second standard test is whether a docking protocol can rank molecules whose potencies are known.
We use exactly the five inhibitors of notebook 5 - same compounds, same SMILES, taken verbatim from
the `INHIBITORS` dictionary there - so that the two notebooks can be compared directly. They are
five different chemical scaffolds, four of them with a co-crystal structure, and four of the five
reached clinical trials.

Remember the practical remark from notebook 5 about where these structures come from: each SMILES is
from the PDB chemical component dictionary or from ChEMBL, cross-checked between the two by
InChIKey, and for SAR405838 and CGM097 the PubChem entry carries *different* stereochemistry, which
we deliberately do not use.

The potencies come from ChEMBL (target CHEMBL5023, human MDM2) and are in the repository, so this
notebook makes no web request. For each compound all binding-assay records with a pChEMBL value were
taken and the median reported, with the range. Look at those ranges before you read anything into a
correlation: nutlin-3a has forty measurements spanning four log units. Pooled literature potencies
are a noisy reference, and that noise sets a ceiling on any correlation we could possibly find.

In [ ]:
# SMILES taken verbatim from the INHIBITORS dictionary of notebook 5.
INHIBITORS = {
    "Nutlin-3a": "COc1ccc(c(OC(C)C)c1)C2=N[C@H]([C@H](N2C(=O)N3CCNC(=O)C3)c4ccc(Cl)cc4)c5ccc(Cl)cc5",
    "SAR405838": "CC(C)(C)C[C@@H]1N[C@@H](C(=O)N[C@H]2CC[C@H](O)CC2)[C@H](c2cccc(Cl)c2F)[C@]12C(=O)Nc1cc(Cl)ccc12",
    "Siremadlin": "COc1ncc(c(OC)n1)c2nc3C(=O)N([C@@H](c4ccc(Cl)cc4)c3n2C(C)C)C5=CC(=CN(C)C5=O)Cl",
    "NVP-CGM097": "COc1cc2c(cc1OC(C)C)[C@H](c1ccc(Cl)cc1)N(c1ccc(N(C)C[C@H]3CC[C@H](N4CCN(C)C(=O)C4)CC3)cc1)C(=O)C2",
    "Navtemadlin": "CC(C)[C@@H](CS(=O)(=O)C(C)C)N1[C@@H]([C@H](C[C@](C1=O)(C)CC(=O)O)C2=CC(=CC=C2)Cl)C3=CC=C(C=C3)Cl",
}

potency = pd.read_csv(os.path.join(NB6_DIR, "known_inhibitors_potency.csv"))
print("Measured potencies against MDM2, from ChEMBL (pChEMBL = -log10 of the potency in M):\n")
potency

In [ ]:
# Dock the five: about 30 s on 8 cores, a minute on 4.
inhibitor_rows = []
for name, smiles in INHIBITORS.items():
    t0 = time.time()
    pdbqt, mol3d, error = prepare_ligand(smiles)
    if pdbqt is None:
        print(f"  {name:12s} could not be prepared: {error}")
        inhibitor_rows.append({"compound": name, "docking score": np.nan})
        continue

    docker.set_ligand_from_string(pdbqt)
    docker.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
    score = docker.energies(n_poses=1)[0][0]

    mol_2d = Chem.MolFromSmiles(smiles)
    inhibitor_rows.append({
        "compound": name,
        "docking score": score,
        "MW": round(Descriptors.MolWt(mol_2d), 1),
        "rotatable bonds": Descriptors.NumRotatableBonds(mol_2d),
        "heavy atoms": mol_2d.GetNumHeavyAtoms(),
    })
    print(f"  {name:12s} score {score:6.2f} kcal/mol   ({time.time() - t0:.1f} s)", flush=True)

inhibitors_df = (pd.DataFrame(inhibitor_rows)
                 .merge(potency[["compound", "median_pchembl", "min_pchembl", "max_pchembl",
                                 "n_measurements"]],
                        on="compound", how="left"))

# A docking score is better when it is more negative, so rank ascending;
# potency is better when pChEMBL is larger.
inhibitors_df["rank by score"] = inhibitors_df["docking score"].rank().astype(int)
inhibitors_df["rank by potency"] = inhibitors_df["median_pchembl"].rank(ascending=False).astype(int)
inhibitors_df["score per heavy atom"] = (inhibitors_df["docking score"]
                                         / inhibitors_df["heavy atoms"]).round(3)
inhibitors_df.sort_values("docking score")

In [ ]:
spearman = inhibitors_df["docking score"].corr(-inhibitors_df["median_pchembl"], method="spearman")
pearson = inhibitors_df["docking score"].corr(-inhibitors_df["median_pchembl"], method="pearson")

print(f"Spearman correlation between docking score and measured potency: {spearman:+.2f}")
print(f"Pearson  correlation                                          : {pearson:+.2f}")
print(f"(over {len(inhibitors_df)} compounds; the sign is arranged so that +1 would mean "
      "'docking ranks them exactly right')")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(inhibitors_df["median_pchembl"], inhibitors_df["docking score"],
           s=70, color="steelblue", zorder=3)
ax.errorbar(inhibitors_df["median_pchembl"], inhibitors_df["docking score"],
            xerr=[inhibitors_df["median_pchembl"] - inhibitors_df["min_pchembl"],
                  inhibitors_df["max_pchembl"] - inhibitors_df["median_pchembl"]],
            fmt="none", ecolor="lightsteelblue", capsize=3, zorder=2)
for _, row in inhibitors_df.iterrows():
    ax.annotate(row["compound"], (row["median_pchembl"], row["docking score"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xlabel("measured potency (pChEMBL; higher = more potent)")
ax.set_ylabel("Vina score (kcal/mol; lower = better)")
ax.set_title(f"Docking score vs measured potency (Spearman {spearman:+.2f})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### What that scatter plot means

The five compounds span roughly **2.5 log units** of potency - a factor of a few hundred in binding
constant - and the docking scores cover barely 2 kcal/mol in an order that bears almost no relation
to the measurements. The horizontal bars are not error bars on a measurement; they are the *range of
what the literature reports* for the same compound.

Read the two correlation coefficients together, and distrust both. The Spearman coefficient of about
+0.2 says the *ranking* is barely better than arbitrary. The Pearson coefficient looks much more
respectable, but it is carried almost entirely by nutlin-3a sitting at the bottom left; with five
points, one compound can manufacture a correlation. Neither number would survive a larger set, and
neither should persuade you of anything - which is itself the lesson about small validation sets.

Look at what the ranking actually does to individual compounds. **Navtemadlin** is one of the two
most potent compounds here - the most potent of all on its ITC and SPR measurements, with a
sub-nanomolar K<sub>d</sub> - and it gets the **worst score of the five**. **Nutlin-3a** is the
weakest binder by more than two orders of magnitude, and it scores second-worst, just above
navtemadlin: the two compounds at opposite ends of the potency range are placed next to each other.
If you used this scoring function to decide which of these five to pursue, you would get it wrong.

Four reasons, all of them general:

1. **The scoring function is crude.** Five fitted terms, no explicit solvent, no entropy beyond a
   torsion count. Published benchmarks put the correlation between Vina scores and affinity across
   diverse compounds at a Pearson r of roughly 0.4-0.5 at best, and much lower within a single
   target - which is the case that actually matters for choosing compounds.
2. **These molecules are hard cases.** 555-660 Da, six to nine rotatable bonds. Vina's torsion
   penalty is a flat count, so it taxes a flexible molecule the same way whether its flexibility
   costs it anything or not.
3. **Charge states are guessed.** Navtemadlin has a carboxylic acid, which at pH 7 is ionised; we
   docked the neutral form that comes out of its SMILES. Its measured binding also involves an
   interaction with His96 that an empirical hydrogen-bond term represents poorly.
4. **The reference numbers are noisy.** Pooling IC50, Ki and Kd from forty papers and different
   assay formats is itself an approximation, as the bars show.

And yet docking is worth doing, as long as you ask it the right question. A docking score is a poor
estimate of *how strongly* a molecule binds, but it is a usable filter for *whether* a molecule can
occupy a site at all: all five of these compounds score in a range that a molecule which simply
cannot fit would never reach. Docking is most useful for discarding the hopeless and for generating
a pose you can criticise - not for ranking the promising.

**Question 8.** The five compounds' docking scores do not follow their potencies.

- Which of the four reasons above do you think matters most here, and would you expect the scoring
  function to do better at ranking close analogues of *one* scaffold than these five unrelated
  molecules?
- Now look at the "score per heavy atom" column, which is the docking equivalent of ligand
  efficiency. Why might dividing by size be a fairer comparison between these compounds - and what
  new way of being misled does it introduce?

## 8. Screening the candidates of notebooks 4 and 5

Now the step the funnel was built for. Notebook 4 reduced a 646,891-compound vendor catalogue to
about 5,300 candidates using properties, fingerprint similarity and three machine-learning models.
Notebook 5 put those through a three-finger pharmacophore and kept the compounds that reproduce the
motif almost perfectly. Those survivors are what we dock.

Three files from the previous notebooks are relevant, and it is worth being explicit about which we
use and why:

| File | What is in it | Used here? |
|---|---|---|
| `scratch/pharmacophore_refined_collection.csv` | the compounds that passed notebook 5's score cut-off, with SMILES and the pharmacophore score | **yes** - this is the set we dock |
| `scratch/pharmacophore_top_hits.sdf` | the top 50 hits as 3D structures *aligned to the pharmacophore* | no - see below |
| `scratch/hits_from_classification_models.sdf` | all ~5,300 compounds from notebook 4 | no - see below |

**Why not the aligned poses?** Because they are aligned to the *pharmacophore*, not placed in the
pocket by an energy function, and Vina randomises the pose before it searches anyway. The only
thing we could usefully take from them is the ligand's internal conformation, and section 6.5 showed
that the conformer matters - which is exactly what exercise 2 asks you to explore.

**Why not all 5,300 compounds from notebook 4?** Honest arithmetic: this notebook docks at about
4.6 seconds per compound on eight cores, so 5,326 compounds is roughly **seven hours**. That does not
fit a two-hour practical. It is not a hard problem - it is an afternoon on a few dozen cores of
CalcUA, submitted as a Slurm array job, and that is how it would be done in practice - but it is not
something to run in a notebook while a class waits. So we dock the full output of notebook 5 and
leave the larger set as exercise 4.

This is the sequencing that makes a funnel work: the expensive method is applied to the compounds
that the cheap methods have already endorsed. It also inherits every bias of those cheap methods,
which is the subject of section 9.3.

In [ ]:
CANDIDATES_CSV = os.path.join(SCRATCH, "pharmacophore_refined_collection.csv")
CANDIDATES_FALLBACK = os.path.join(NB6_DIR, "pharmacophore_refined_collection_reference.csv")
NB4_SDF = os.path.join(SCRATCH, "hits_from_classification_models.sdf")


def load_candidates():
    """Read the compounds that survived the pharmacophore screen of notebook 5.

    If notebook 5 has not been run on this machine, fall back to the committed
    copy of the reference run, so that this notebook still works - but say so
    loudly, because the numbers will then not be the student's own.
    """
    if os.path.exists(CANDIDATES_CSV):
        df = pd.read_csv(CANDIDATES_CSV)
        print(f"Reading {CANDIDATES_CSV} (the output of notebook 5): {len(df)} compounds")
        return df, "your own notebook 5 run"

    print(f"{CANDIDATES_CSV} not found.")
    print("  => Please run notebooks 4 and 5 first: this notebook is meant to dock *your*")
    print("     candidate set, and the discussion in section 9 compares the two screens.")
    print(f"  => Falling back to the committed reference run, {CANDIDATES_FALLBACK},")
    print("     so that the rest of this notebook can still be executed.")
    df = pd.read_csv(CANDIDATES_FALLBACK)
    print(f"     {len(df)} compounds read from the reference run.")
    return df, "the committed reference run"


candidates, candidates_source = load_candidates()

if os.path.exists(NB4_SDF):
    print(f"\n(for reference: {NB4_SDF} from notebook 4 is also present)")

print(f"\ncolumns: {list(candidates.columns)}")
candidates.head()

### 8.1 Running the screen

**This is the slow cell of the notebook** - a measured 8.9 minutes on 8 cores for the reference set
of 116 compounds, about 13 minutes on 4. Start it, then read section 9 while it runs.

Three things in the loop are worth noticing, because they are what makes a screen survive contact
with real data:

- every compound goes through the same `prepare_ligand()` as the validation compounds, and failures
  are **counted and reported** rather than silently dropped;
- the receptor and its grids are set up **once**, outside the loop - recomputing them per compound
  would double the cost;
- all nine poses of every compound are written to an SDF file, so the poses can be inspected later in
  PyMOL without docking anything again.

In [ ]:
# SLOW: measured at 8.9 minutes on 8 cores for 116 compounds (13 min on 4 cores).
POSES_SDF = os.path.join(SCRATCH, "docking_poses.sdf")

screen_docker = make_docker(RECEPTOR_DRY)
writer = Chem.SDWriter(POSES_SDF)

screen_rows = []
failures = []
t_start = time.time()

for _, row in tqdm(list(candidates.iterrows()), total=len(candidates), desc="Docking"):
    pdbqt, mol3d, error = prepare_ligand(row["smiles"])
    if pdbqt is None:
        failures.append((row["compound_id"], error))
        continue

    try:
        screen_docker.set_ligand_from_string(pdbqt)
        screen_docker.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
        pose_energies = screen_docker.energies(n_poses=N_POSES)
        score = float(pose_energies[0][0])
        poses = poses_to_mol(screen_docker)
    except Exception as exc:                 # a docking run can fail on its own
        failures.append((row["compound_id"], f"docking failed: {exc}"))
        continue

    screen_rows.append({"compound_id": row["compound_id"],
                        "smiles": row["smiles"],
                        "docking_score": score,
                        "tversky_ref": row["tversky_ref"],
                        "mw": row["mw"],
                        "clogp": row["clogp"]})

    # keep every pose, with its score, for later inspection
    for conf_id in range(poses.GetNumConformers()):
        pose = Chem.Mol(poses, confId=conf_id)
        pose.SetProp("_Name", f"{row['compound_id']}_pose{conf_id + 1}")
        pose.SetProp("compound_id", str(row["compound_id"]))
        pose.SetProp("pose", str(conf_id + 1))
        pose.SetProp("vina_score", f"{pose_energies[conf_id][0]:.3f}")
        pose.SetProp("tversky_ref", f"{row['tversky_ref']}")
        writer.write(pose, confId=conf_id)

writer.close()
screen_time = time.time() - t_start

docked = pd.DataFrame(screen_rows).sort_values("docking_score").reset_index(drop=True)
print(f"\n{len(docked)} of {len(candidates)} compounds produced a pose "
      f"in {screen_time / 60:.1f} minutes on {N_CPU} cores")
print(f"  that is {screen_time / max(len(candidates), 1):.1f} s per compound")
print(f"  {len(failures)} compounds failed:")
for compound_id, error in failures[:10]:
    print(f"    {compound_id}: {error}")
print(f"\nall poses written to {POSES_SDF}")

## 9. Analysing the hits

### 9.1 How are the scores distributed?

The first thing to look at is the distribution, and the first thing to notice about it is that
**there is no cut-off in it**. Unlike the pharmacophore screen, which either matched a molecule or
did not, docking gives every compound a number. Every compound "docks". Deciding what counts as a
hit is entirely our choice, and the only honest way to make that choice is to put the distribution
next to a reference - which is what the five known inhibitors of section 7 are for.

In [ ]:
scores = docked["docking_score"].to_numpy()
known_scores = inhibitors_df["docking score"].dropna().to_numpy()

print(f"compounds docked                      : {len(scores)}")
print(f"score  min / median / max             : {scores.min():.2f} / {np.median(scores):.2f} / {scores.max():.2f}")
print(f"the five known inhibitors scored      : {known_scores.min():.2f} to {known_scores.max():.2f}")
print(f"candidates scoring at least as well as")
print(f"  the best known inhibitor ({known_scores.min():.2f})     : "
      f"{int((scores <= known_scores.min()).sum())}")
print(f"  the median known inhibitor ({np.median(known_scores):.2f})   : "
      f"{int((scores <= np.median(known_scores)).sum())}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(scores, bins=25, color="steelblue", edgecolor="white")
for s in known_scores:
    ax[0].axvline(s, color="firebrick", lw=1.2, alpha=0.8)
ax[0].set_xlabel("Vina score (kcal/mol)")
ax[0].set_ylabel("number of compounds")
ax[0].set_title("Docking scores of the candidate set\n(red lines: the five known inhibitors)")

ax[1].plot(np.arange(1, len(scores) + 1), scores, color="steelblue")
ax[1].axhline(known_scores.min(), color="firebrick", ls="--", lw=1.2,
              label=f"best known inhibitor ({known_scores.min():.2f})")
ax[1].set_xlabel("rank")
ax[1].set_ylabel("Vina score (kcal/mol)")
ax[1].set_title("The ranking")
ax[1].legend()

plt.tight_layout()
plt.show()

About half of the candidates score better than the *median* of the five known inhibitors, and a
handful beat the best of them. It is worth being very clear about how little that means. These
compounds were pre-selected by notebook 4 to be large and lipophilic, and by notebook 5 to place
three hydrophobic groups the way p53 does - and Vina's score rises with the amount of buried apolar
surface. A big greasy molecule in a big greasy pocket scores well almost by construction. Section 7
showed that this score cannot even rank five known inhibitors correctly, so it certainly cannot tell
us that a catalogue compound beats nutlin-3a.

### 9.2 The top-ranked compounds

In [ ]:
N_SHOW = 20

top = docked.head(N_SHOW).copy()
top.insert(0, "rank", np.arange(1, len(top) + 1))
top[["rank", "compound_id", "docking_score", "tversky_ref", "mw", "clogp"]]

In [ ]:
top_mols, top_legends = [], []
for _, row in top.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"])
    if mol is None:
        continue
    rdDepictor.Compute2DCoords(mol)
    top_mols.append(mol)
    top_legends.append(f"{row['compound_id']}\n{row['docking_score']:.2f} kcal/mol "
                       f"(pharm. {row['tversky_ref']:.2f})")

Draw.MolsToGridImage(top_mols, legends=top_legends, molsPerRow=4, subImgSize=(300, 280))

This is the part of a virtual screen that cannot be automated: somebody has to look. Ask of each of
these structures whether it is a molecule you would want to make or buy, whether anything in it
looks reactive or promiscuous, and whether the scaffolds are different enough from one another that
the hit list is really twenty ideas rather than two.

In [ ]:
# The best-scoring compound in its predicted pose, next to the crystallographic
# nutlin-3a it is supposed to displace.
best_hit = docked.iloc[0]
print(f"best-scoring candidate: {best_hit['compound_id']}  "
      f"{best_hit['docking_score']:.2f} kcal/mol, pharmacophore score {best_hit['tversky_ref']:.2f}")

# re-dock it so we have the pose in memory (a couple of seconds)
pdbqt, _, error = prepare_ligand(best_hit["smiles"])
screen_docker.set_ligand_from_string(pdbqt)
screen_docker.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
best_pose = poses_to_mol(screen_docker)

view = py3Dmol.view(width=800, height=550)
view.addModel(pdb_4hg7, "pdb")
view.setStyle({"model": 0}, {})
view.setStyle({"model": 0, "hetflag": False}, {"cartoon": {"color": "lightgray"}})
view.addSurface(py3Dmol.VDW, {"opacity": 0.55, "color": "lightgray"},
                {"model": 0, "hetflag": False})
view.addStyle({"model": 0, "resn": LIGAND_RESNAME},
              {"stick": {"colorscheme": "yellowCarbon", "radius": 0.18}})
view.addStyle({"model": 0, "resi": POCKET_RESIDUES},
              {"stick": {"colorscheme": "whiteCarbon", "radius": 0.12}})

view.addModel(Chem.MolToPDBBlock(Chem.RemoveHs(best_pose), confId=0), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})

view.zoomTo({"model": 0, "resn": LIGAND_RESNAME})
view.setBackgroundColor("white")
view.show()

Cyan is the predicted pose of the best-scoring candidate, yellow is nutlin-3a where the crystal
structure puts it. Rotate it and apply the visual triage of notebook 5: is the molecule inside the
cleft, or does half of it stick into the solvent? Are its hydrophobic groups in the three subpockets
that nutlin-3a occupies, or somewhere else altogether? Remember while you look that in section 6 a
pose exactly like this one, for a molecule we *know* binds, was 8 A wrong.

### 9.3 Do the two screens agree?

Here is a question the funnel lets us ask and that is worth more than either ranking on its own.
Notebook 5 ranked these compounds by pharmacophore overlap (`tversky_ref`) and this notebook ranks
them by docking score. The two methods use different information: one knows three feature positions
derived from p53, the other knows 911 receptor atoms and an energy-like function. How much do they
agree?

In [ ]:
rho = docked["docking_score"].corr(-docked["tversky_ref"], method="spearman")
print(f"Spearman correlation between the docking rank and the pharmacophore rank: {rho:+.2f}")
print("(+1 would mean the two methods order the compounds identically)")

# how much do the two top-20 lists overlap?
top20_docking = set(docked.head(20)["compound_id"])
top20_pharmacophore = set(docked.sort_values("tversky_ref", ascending=False).head(20)["compound_id"])
overlap = top20_docking & top20_pharmacophore
print(f"\ncompounds in both top-20 lists: {len(overlap)} of 20")
print("  ", sorted(overlap) if overlap else "none")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(docked["tversky_ref"], docked["docking_score"], s=25, alpha=0.7, color="steelblue")
ax.set_xlabel("pharmacophore score from notebook 5 (tversky_ref; higher = better)")
ax.set_ylabel("Vina score (kcal/mol; lower = better)")
ax.set_title(f"Two independent screens of the same compounds (Spearman {rho:+.2f})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The two rankings are essentially **uncorrelated**. That is not a contradiction and it is not a sign
that one of them is broken - it is the expected result, and it has a useful reading and a sobering
one.

The useful reading: the methods are genuinely independent, so a compound that scores well on *both*
has two unrelated arguments in its favour. That is the real product of a funnel, and it is why the
intersection of the two top-20 lists is a more interesting set than either list alone.

The sobering reading: two defensible methods applied to the same 116 molecules and the same protein
disagree almost completely about which are the best ones. They cannot both be good rankings of the
same underlying truth, and neither has been shown to be a good ranking of anything at all, because
no compound in this notebook has ever been near an assay. The honest summary of a virtual screen is not "we found the best compounds" but "we
produced a shortlist, by criteria we can state, and we know roughly how unreliable each criterion
is."

**Question 9.** You can afford to buy and test ten of these 116 compounds. Would you take the ten
best by docking score, the ten best by pharmacophore score, or the ten with the best *combined*
ranking - and what would you sacrifice in each case? Which is the stronger evidence that a compound
binds: a high pharmacophore score, a good docking score, or agreement between the two?

**Question 10.** Everything in this notebook has been applied to compounds that earlier notebooks
already selected as *likely* binders, so we have no idea how often this protocol would endorse a
molecule picked at random. How would you measure the **false-positive rate** of this docking screen?
What would you have to dock, how would you choose it, and what would you expect to see if the screen
were worthless? (Search for "decoys" and "DUD-E" if you want to see how this is done properly.)

### 9.4 Saving the result and extending the funnel

In [ ]:
RANKED_CSV = os.path.join(SCRATCH, "docking_ranked_collection.csv")

docked["rank_docking"] = np.arange(1, len(docked) + 1)
docked["rank_pharmacophore"] = docked["tversky_ref"].rank(ascending=False).astype(int)
docked["rank_combined"] = (docked["rank_docking"] + docked["rank_pharmacophore"]) / 2
docked.sort_values("rank_combined").to_csv(RANKED_CSV, index=False)
print(f"{len(docked)} compounds written to {RANKED_CSV}, ranked by the two methods combined\n")

print("The ten compounds that both methods like:")
docked.sort_values("rank_combined").head(10)[
    ["compound_id", "docking_score", "tversky_ref", "rank_docking", "rank_pharmacophore", "mw"]
]

In [ ]:
# The screening funnel of notebook 5, with the docking step added.
LIBRARY_SIZE = 646891
NB4_SIZE = 5326          # compounds in hits_from_classification_models.sdf (reference run)

better_than_best_known = int((scores <= known_scores.min()).sum())

funnel = pd.DataFrame([
    ("Asinex vendor library", LIBRARY_SIZE, "-"),
    ("property filter + similarity + ML (notebook 4)", NB4_SIZE,
     "properties and 2D fingerprints, ligand-based"),
    ("three-finger pharmacophore, score >= 0.90 (notebook 5)", len(candidates),
     "3D features from the p53 peptide, structure-based"),
    ("docked successfully into 4HG7 (this notebook)", len(docked),
     "all-atom receptor and an empirical score"),
    (f"score at least as good as the best known inhibitor ({known_scores.min():.2f})",
     better_than_best_known, "all-atom receptor and an empirical score"),
], columns=["step", "compounds", "information used"])

funnel["% of library"] = (100.0 * funnel["compounds"] / LIBRARY_SIZE).round(4)
funnel

Four orders of magnitude, from a vendor catalogue to a handful of compounds, each step using
information the previous one did not have. Two remarks, in the spirit of notebook 5's version of
this table.

The last row is the one to distrust most. "Scores at least as well as a known nanomolar inhibitor"
sounds like a strong statement and is almost an empty one, for the reason given in section 9.1: the
score rewards size and greasiness, and every compound in this set was selected for size and
greasiness. A threshold taken from the known actives is a *reasonable* way to choose a shortlist,
not evidence about any individual compound.

And note what has *not* narrowed: every compound here is a descendant of the notebook 4 property
filter, so nothing lighter than about 450 Da can appear, whatever its merits. A funnel can only ever
remove; it cannot recover what an earlier step discarded. The one chance a late step has to correct
an early one is the kind of disagreement we found in section 9.3 - which is a reason to take such
disagreements seriously rather than to average them away.

## 10. What a good docking score is not

Notebook 5 ended with the statement that a pharmacophore hit is a hypothesis and not a binder. The
same statement has to be made about docking, and it has to be made more firmly, because a docking
result is so much more *persuasive*. It comes with a picture of a molecule nestled in a pocket and a
number in kcal/mol. Both invite more confidence than they deserve.

**A good docking score is not evidence that a compound binds.** What this notebook has actually
demonstrated, with its own numbers, is:

- the scoring function could not distinguish the experimental pose of nutlin-3a from a pose 8 A away
  (section 6.3);
- it ranked five known inhibitors spanning 2.5 log units of potency in an order essentially
  unrelated to their potencies, putting the weakest and one of the strongest side by side
  (section 7);
- it scores about half of a pre-filtered catalogue above the median of five clinical compounds
  (section 9.1);
- and it disagrees almost completely with the other structure-based method we applied to the same
  molecules (section 9.3).

What docking *is* good for, and it is genuinely useful:

- **generating a structural hypothesis.** A pose is a concrete, criticisable claim about how a
  molecule would sit in a site. It tells a chemist which group to modify and why, and it can be
  wrong in an informative way.
- **ruling things out.** A molecule that cannot be placed in the site without a severe clash is
  unlikely to bind there. Negative predictions are the more reliable half of docking.
- **enriching a shortlist.** Over thousands of compounds, docking moves actives up the list more
  often than chance - modestly, but enough to pay for itself when the alternative is buying at
  random.

What would have to happen before anyone claimed that one of our 116 compounds binds MDM2, roughly
in order of cost:

1. **Rescoring.** Score the same poses with independent functions - Vinardo, an MM-GBSA
   calculation, or a machine-learned function - and keep the compounds that survive all of them.
   Agreement between scoring functions is weak evidence, but it is cheap.
2. **Pose plausibility.** Check the geometry itself: internal strain, stereochemistry, clashes with
   the receptor. The `posebusters` package automates these checks and is worth knowing about; a pose
   that fails them should be discarded whatever it scores.
3. **Molecular dynamics.** Simulate the complex for tens or hundreds of nanoseconds, with explicit
   water and a flexible protein, and see whether the pose survives. Many do not.
4. **Free-energy calculations.** Thermodynamic integration or FEP on a handful of the best poses -
   the most accurate computational estimate available, and expensive enough that you would only do it
   for a few compounds.
5. **An assay.** For MDM2 this means a fluorescence-polarisation or HTRF experiment measuring
   displacement of a p53 peptide from MDM2, as described in notebooks 4 and 5.

Only step 5 produces evidence. Everything before it - including everything in this notebook -
produces a shortlist of things worth spending step 5 on. That is a real and valuable contribution,
and it is a different thing from a discovery.

## 11. Exercises

Work through at least two of these. All of them are small modifications to cells you have already
run; the point of each is a concept, not the code.

1. **The box.** Re-run the redocking of section 6.1 with `BOX_MARGIN = 2.5` and with
   `BOX_MARGIN = 10.0` (re-run the cells of section 4 and 5 after changing it, so the grids are
   rebuilt). How do the score and the RMSD change, and how does the time change? Does a bigger box
   help or hurt, and why? This is the check that lets you rule the box out as a cause of the
   section 6 failure.

2. **Several starting conformers.** Section 6.5 showed that the ligand's internal geometry is part
   of the protocol. Use `prepare_ligand(..., n_conformers=10, conformer_id=i)` in a loop over
   `i` to dock ten different starting conformers of nutlin-3a, and keep the best-scoring result over
   all of them. Does that recover the crystal pose without using crystal information? What does it do
   to the cost per compound, and would you spend it in a screen?

3. **A different scoring function.** Vina ships a second function, Vinardo, trained differently:
   `Vina(sf_name="vinardo", ...)`. Repeat section 7 with it. Do the five known inhibitors rank any
   better? Do the *same* compounds come out on top in section 8? If two scoring functions disagree
   about which compounds are best, what have you learned about either?

4. **The compounds we skipped.** Dock a random sample of 200 compounds from
   `scratch/hits_from_classification_models.sdf` - the notebook 4 output, which has *not* been
   through the pharmacophore filter. Do they score worse than the 116 that passed it? If they score
   just as well, what does that say about what the pharmacophore filter accomplished, and about what
   the docking score measures?

5. **Decoys, properly.** Following on from question 10: take 200 compounds from the Asinex library
   of notebook 4 *without* applying the property filter, dock them, and compare their score
   distribution with the one in section 9.1. How much of the apparent quality of our hits is
   explained by molecular weight alone? Try plotting the docking score against molecular weight for
   all the compounds you have docked.

6. **Pose plausibility.** Install `posebusters` (`pip install posebusters`) and run its `redock`
   checks on the redocking of section 6 and its `dock` checks on your top poses from section 8. Which
   checks does each pose pass? Does any compound in your top 20 produce a geometrically implausible
   pose despite a good score?

## 12. Take-home messages

1. Docking is a **search** over ligand poses plus a **scoring function**, and the two fail
   independently. Distinguishing a search failure from a scoring failure is the first thing to do
   when a docking result looks wrong, and it is diagnosable: raise `exhaustiveness`, and score the
   experimental pose directly.
2. `exhaustiveness` buys **confidence in the search**, not accuracy in the score. Eight times the
   effort found the same wrong pose eight times more reliably.
3. **Receptor preparation is where campaigns are lost.** Chain, alternate conformations, ions,
   histidine tautomers - and above all the **waters**. Three deleted waters cost about a kcal/mol in
   the native pose of nutlin-3a, which is large next to the hundredths of a kcal/mol that separated
   the right pose from an 8 A error - and each of those waters is displaced by at least one other
   MDM2 inhibitor, so there is no setting that is right for every ligand.
4. Docking needs a **defined site**. The box is derived here from a crystallographic ligand, which
   quietly assumes we already know where the site is; its size is a genuine trade-off between
   forbidding the right answer and diluting the search.
5. **Always redock.** It is the easiest test a protocol can be given - the receptor was solved with
   that very ligand in it - and ours failed it at 8 A. Without that check we would have screened with
   a broken protocol and never known.
6. **The ligand conformer you hand in is part of your protocol.** Vina varies the torsions but
   freezes rings, bond angles and non-rotatable geometry. For unknown compounds there is no crystal
   conformation to fall back on, which is a limitation of every screen of this kind.
7. A **docking score is a weak predictor of affinity**. Five known inhibitors spanning 2.5 log units
   came out in an order essentially unrelated to their potencies (Spearman about +0.2 over five
   compounds, which is to say: nothing). Docking discriminates "could occupy this site" from "could
   not" far better than "binds tightly" from "binds weakly".
8. Docking and pharmacophore searching **disagree**, and that is their value: two independent
   arguments are worth more than one, and their disagreement is a more honest description of our
   uncertainty than either ranking alone.
9. **A good docking score is not evidence of binding.** Rescoring, pose checks, MD and free-energy
   methods raise the confidence; only an assay produces evidence.

## 13. References

**Docking methodology and the software used**

1. Trott, O.; Olson, A.J. (2010) AutoDock Vina: improving the speed and accuracy of docking with a new scoring function, efficient optimization, and multithreading. *J. Comput. Chem.* **31**, 455-461. <a href="https://doi.org/10.1002/jcc.21334" target="_blank">doi:10.1002/jcc.21334</a> - **the reference to cite when you use Vina**; the Monte-Carlo search and the empirical scoring function of section 1.
2. Eberhardt, J.; Santos-Martins, D.; Tillack, A.F.; Forli, S. (2021) AutoDock Vina 1.2.0: new docking methods, expanded force field, and Python bindings. *J. Chem. Inf. Model.* **61**, 3891-3898. <a href="https://doi.org/10.1021/acs.jcim.1c00203" target="_blank">doi:10.1021/acs.jcim.1c00203</a> - the version and the Python API used here, and the reference for **Meeko**.
3. Morris, G.M.; Huey, R.; Lindstrom, W.; Sanner, M.F.; Belew, R.K.; Goodsell, D.S.; Olson, A.J. (2009) AutoDock4 and AutoDockTools4: automated docking with selective receptor flexibility. *J. Comput. Chem.* **30**, 2785-2791. <a href="https://doi.org/10.1002/jcc.21256" target="_blank">doi:10.1002/jcc.21256</a> - the PDBQT format and the atom types of section 3.4.
4. Quiroga, R.; Villarreal, M.A. (2016) Vinardo: a scoring function based on Autodock Vina improves scoring, docking, and virtual screening. *PLoS ONE* **11**, e0155183. <a href="https://doi.org/10.1371/journal.pone.0155183" target="_blank">doi:10.1371/journal.pone.0155183</a> - the alternative scoring function of exercise 3.
5. Buttenschoen, M.; Morris, G.M.; Deane, C.M. (2024) PoseBusters: AI-based docking methods fail to generate physically valid poses or generalise to novel sequences. *Chem. Sci.* **15**, 3130-3139. <a href="https://doi.org/10.1039/D3SC04185A" target="_blank">doi:10.1039/D3SC04185A</a> - the pose-plausibility checks of section 10 and exercise 6.
6. Landrum, G. *et al.* RDKit: open-source cheminformatics. <a href="https://www.rdkit.org" target="_blank">https://www.rdkit.org</a> - the conformer generation, the symmetry-corrected RMSD and the drawing in this notebook.

**How well docking actually works**

7. Warren, G.L.; Andrews, C.W.; Capelli, A.-M.; Clarke, B.; LaLonde, J.; Lambert, M.H.; Lindvall, M.; Nevins, N.; Semus, S.F.; Senger, S. *et al.* (2006) A critical assessment of docking programs and scoring functions. *J. Med. Chem.* **49**, 5912-5931. <a href="https://doi.org/10.1021/jm050362n" target="_blank">doi:10.1021/jm050362n</a> - the classic survey: poses often right, affinities essentially not predicted.
8. Chen, Y.-C. (2015) Beware of docking! *Trends Pharmacol. Sci.* **36**, 78-95. <a href="https://doi.org/10.1016/j.tips.2014.12.001" target="_blank">doi:10.1016/j.tips.2014.12.001</a> - a readable catalogue of the ways a docking study goes wrong, including most of section 6.
9. Mysinger, M.M.; Carchia, M.; Irwin, J.J.; Shoichet, B.K. (2012) Directory of useful decoys, enhanced (DUD-E): better ligands and decoys for better benchmarking. *J. Med. Chem.* **55**, 6582-6594. <a href="https://doi.org/10.1021/jm300687e" target="_blank">doi:10.1021/jm300687e</a> - property-matched decoys, for question 10 and exercise 5.
10. Su, M.; Yang, Q.; Du, Y.; Feng, G.; Liu, Z.; Li, Y.; Wang, R. (2019) Comparative assessment of scoring functions: the CASF-2016 update. *J. Chem. Inf. Model.* **59**, 895-913. <a href="https://doi.org/10.1021/acs.jcim.8b00545" target="_blank">doi:10.1021/acs.jcim.8b00545</a> - the benchmark the correlation figures quoted in section 7 come from.
11. Amaro, R.E.; Baudry, J.; Chodera, J.; Demir, O.; McCammon, J.A.; Miao, Y.; Smith, J.C. (2018) Ensemble docking in drug discovery. *Biophys. J.* **114**, 2271-2278. <a href="https://doi.org/10.1016/j.bpj.2018.02.038" target="_blank">doi:10.1016/j.bpj.2018.02.038</a> - what to do about the rigid receptor of question 6.

**Water in binding sites**

12. Ladbury, J.E. (1996) Just add water! The effect of water on the specificity of protein-ligand binding sites and its potential application to drug design. *Chem. Biol.* **3**, 973-980. <a href="https://doi.org/10.1016/S1074-5521(96)90164-7" target="_blank">doi:10.1016/S1074-5521(96)90164-7</a> - the problem of section 6.4.
13. Bodnarchuk, M.S. (2016) Water, water, everywhere... It's time to stop and think. *Drug Discov. Today* **21**, 1139-1146. <a href="https://doi.org/10.1016/j.drudis.2016.05.009" target="_blank">doi:10.1016/j.drudis.2016.05.009</a> - how bridging waters are handled in practice.

**MDM2: the target, the structures and the inhibitors** (as in notebook 5)

14. Kussie, P.H.; Gorina, S.; Marechal, V.; Elenbaas, B.; Moreau, J.; Levine, A.J.; Pavletich, N.P. (1996) Structure of the MDM2 oncoprotein bound to the p53 tumor suppressor transactivation domain. *Science* **274**, 948-953. <a href="https://doi.org/10.1126/science.274.5289.948" target="_blank">doi:10.1126/science.274.5289.948</a> - PDB <a href="https://www.rcsb.org/structure/1YCR" target="_blank">1YCR</a>, the hot spots of section 4.1.
15. Anil, B.; Riedinger, C.; Endicott, J.A.; Noble, M.E.M. (2013) The structure of an MDM2-Nutlin-3a complex solved by the use of a validated MDM2 surface-entropy reduction mutant. *Acta Crystallogr. D* **69**, 1358-1366. <a href="https://doi.org/10.1107/S0907444913004459" target="_blank">doi:10.1107/S0907444913004459</a> - PDB <a href="https://www.rcsb.org/structure/4HG7" target="_blank">4HG7</a>, **the receptor of this notebook**, including the E69A/K70A surface-entropy reduction mutations mentioned in section 3.1.
16. Vassilev, L.T.; Vu, B.T.; Graves, B.; Carvajal, D.; Podlaski, F.; Filipovic, Z.; Kong, N.; Kammlott, U.; Lukacs, C.; Klein, C.; Fotouhi, N.; Liu, E.A. (2004) In vivo activation of the p53 pathway by small-molecule antagonists of MDM2. *Science* **303**, 844-848. <a href="https://doi.org/10.1126/science.1092472" target="_blank">doi:10.1126/science.1092472</a> - nutlin-3a.
17. Estrada-Ortiz, N.; Neochoritis, C.G.; Doemling, A. (2016) How to design a successful p53-MDM2/X interaction inhibitor: a thorough overview based on crystal structures. *ChemMedChem* **11**, 757-772. <a href="https://doi.org/10.1002/cmdc.201500487" target="_blank">doi:10.1002/cmdc.201500487</a> - the binding modes of all the inhibitor classes of section 7, and the Tyr100 movement of question 6.
18. Wang, S.; Sun, W.; Zhao, Y.; McEachern, D.; Meaux, I.; Barriere, C.; Stuckey, J.A.; Meagher, J.L.; Bai, L.; Liu, L. *et al.* (2014) SAR405838: an optimized inhibitor of MDM2-p53 interaction that induces complete and durable tumor regression. *Cancer Res.* **74**, 5855-5865. <a href="https://doi.org/10.1158/0008-5472.CAN-14-0799" target="_blank">doi:10.1158/0008-5472.CAN-14-0799</a> - PDB <a href="https://www.rcsb.org/structure/5TRF" target="_blank">5TRF</a>, used in the water comparison of section 6.4.
19. Holzer, P.; Masuya, K.; Furet, P.; Kallen, J.; Valat-Stachyra, T.; Ferretti, S.; Berghausen, J.; Bouisset-Leonard, M.; Buschmann, N.; Pissot-Soldermann, C. *et al.* (2015) Discovery of a dihydroisoquinolinone derivative (NVP-CGM097): a highly potent and selective MDM2 inhibitor undergoing phase 1 clinical trials. *J. Med. Chem.* **58**, 6348-6358. <a href="https://doi.org/10.1021/acs.jmedchem.5b00810" target="_blank">doi:10.1021/acs.jmedchem.5b00810</a> - PDB <a href="https://www.rcsb.org/structure/4ZYF" target="_blank">4ZYF</a>.
20. Jeay, S.; Ferretti, S.; Holzer, P.; Fuchs, J.; Chapeau, E.A.; Wartmann, M.; Sterker, D.; Romanet, V.; Murakami, M.; Kerr, G. *et al.* (2018) Dose and schedule determine distinct molecular mechanisms underlying the efficacy of the p53-MDM2 inhibitor HDM201. *Cancer Res.* **78**, 6257-6267. <a href="https://doi.org/10.1158/0008-5472.CAN-18-0338" target="_blank">doi:10.1158/0008-5472.CAN-18-0338</a> - siremadlin; PDB <a href="https://www.rcsb.org/structure/5OC8" target="_blank">5OC8</a>.
21. Sun, D.; Li, Z.; Rew, Y.; Gribble, M.; Bartberger, M.D.; Beck, H.P.; Canon, J.; Chen, A.; Chen, X.; Chow, D. *et al.* (2014) Discovery of AMG 232, a potent, selective, and orally bioavailable MDM2-p53 inhibitor in clinical development. *J. Med. Chem.* **57**, 1454-1472. <a href="https://doi.org/10.1021/jm401753e" target="_blank">doi:10.1021/jm401753e</a> - navtemadlin; the piperidinone series of PDB <a href="https://www.rcsb.org/structure/4OAS" target="_blank">4OAS</a>, also used in section 6.4.
22. Zhao, Y.; Aguilar, A.; Bernard, D.; Wang, S. (2015) Small-molecule inhibitors of the MDM2-p53 protein-protein interaction (MDM2 inhibitors) in clinical trials for cancer treatment. *J. Med. Chem.* **58**, 1038-1052. <a href="https://doi.org/10.1021/jm501092z" target="_blank">doi:10.1021/jm501092z</a>

**Databases**

23. Zdrazil, B.; Felix, E.; Hunter, F.; Manners, E.J.; Blackshaw, J.; Corbett, S.; de Veij, M.; Ioannidis, H.; Mendez Lopez, D.; Mosquera, J.F. *et al.* (2024) The ChEMBL database in 2023. *Nucleic Acids Res.* **52**, D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004" target="_blank">doi:10.1093/nar/gkad1004</a> - the source of the potencies in section 7.
24. Burley, S.K. *et al.* (2023) RCSB Protein Data Bank. *Nucleic Acids Res.* **51**, D488-D508. <a href="https://doi.org/10.1093/nar/gkac1077" target="_blank">doi:10.1093/nar/gkac1077</a>

## Finished?

You have completed the structure-based half of the workflow this course set out to demonstrate. From
a target and its crystal structures (notebook 1), through the representation of small molecules
(notebook 2), machine-learning models (notebook 3), a focussed library drawn from a vendor catalogue
(notebook 4) and a 3D pharmacophore filter (notebook 5), to a docked, posed and doubly-ranked
shortlist in `scratch/docking_ranked_collection.csv`, with every pose in `scratch/docking_poses.sdf`
for inspection in PyMOL.

What you should take away from this notebook, more than any of the numbers in it, is the habit of
validating a method before trusting it. The redocking test cost a few minutes and told us that our
protocol put a known ligand 8 A from where it belongs. Working out why took the rest of section 6
and taught us more about docking than a successful run would have: that the search was fine, that
three deleted waters mattered more than the score gap between right and wrong, that the conformer we
hand in is part of the protocol,
and that the scoring function could not tell right from wrong even in the easiest case it will ever
be given.

That is not a reason to abandon docking. It is the reason to report what it says with the
uncertainty it deserves - and to keep the last step of the funnel, the one with the pipette, firmly
in view.